<a href="https://colab.research.google.com/github/Leanhchudang2511/baitaptrituenhantao/blob/main/Grab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install streamlit pyngrok anthropic folium streamlit-folium requests -q

✅ Cài xong!


In [2]:
import os
os.makedirs(".streamlit", exist_ok=True)

with open(".streamlit/secrets.toml", "w") as f:
    f.write('ANTHROPIC_API_KEY = ""\n')

✅ Đã tạo secrets.toml!


In [10]:
%%writefile app.py
import streamlit as st
import anthropic
import random
import time
import folium
import requests
import math
from folium.plugins import LocateControl, AntPath
from streamlit_folium import st_folium
from datetime import datetime

st.set_page_config(
    page_title="Grab VN",
    page_icon="🚕",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
@import url(https://fonts.googleapis.com/css2?family=Be+Vietnam+Pro:wght@400;500;600;700;800&display=swap);

html, body, [class*=css] {
    font-family: 'Be Vietnam Pro', sans-serif;
    color: #e8f5e9 !important;
}
.stApp { background: #0a0e0b; }

p, span, label, div, li, td, th, small,
.stMarkdown, .stText,
[data-testid="stMarkdownContainer"] p,
[data-testid="stMarkdownContainer"] span,
.element-container p,
.element-container span {
    color: #e8f5e9 !important;
}
h1, h2, h3, h4, h5, h6 { color: #ffffff !important; }

.stSelectbox > div > div,
.stSelectbox > div > div > div,
div[data-baseweb="select"],
div[data-baseweb="select"] > div,
div[data-baseweb="select"] span,
div[data-baseweb="select"] input,
div[data-baseweb="select"] > div > div,
div[data-baseweb="select"] > div > div > div,
[data-testid="stSelectbox"] div,
[data-testid="stSelectbox"] span {
    background-color: #1a2a1a !important;
    color: #e8f5e9 !important;
    border-color: rgba(0,177,79,0.4) !important;
}

div[data-baseweb="select"] [data-testid="stMarkdownContainer"],
div[data-baseweb="select"] *:not(svg) {
    color: #e8f5e9 !important;
    background-color: transparent !important;
}

div[data-baseweb="popover"],
div[data-baseweb="popover"] > div,
ul[data-baseweb="menu"],
li[data-baseweb="menu-item"],
div[role="listbox"],
div[role="option"],
[data-baseweb="menu"] {
    background: #1a2a1a !important;
    background-color: #1a2a1a !important;
    color: #e8f5e9 !important;
}
li[data-baseweb="menu-item"]:hover,
div[role="option"]:hover {
    background: rgba(0,177,79,0.25) !important;
}
li[data-baseweb="menu-item"] *,
div[role="option"] *,
ul[data-baseweb="menu"] * {
    color: #e8f5e9 !important;
    background-color: transparent !important;
}

div[data-baseweb="select"] svg {
    fill: #a5d6a7 !important;
}

.stTextInput > div > div > input,
.stTextInput > div > div > input::placeholder,
.stChatInputContainer textarea,
.stChatInputContainer textarea::placeholder {
    background: #1e2a1e !important;
    border: 1px solid rgba(0,177,79,0.4) !important;
    border-radius: 10px !important;
    color: #ffffff !important;
    font-size: 15px !important;
}
.stTextInput > div > div > input::placeholder,
.stChatInputContainer textarea::placeholder {
    color: #6a8f6a !important;
}
.stTextInput label, .stSelectbox label { color: #c8e6c9 !important; }

.stNumberInput > div > div > input {
    background: #1e2a1e !important;
    color: #e8f5e9 !important;
    border-color: rgba(0,177,79,0.4) !important;
}

.stSlider > div > div > div > div {
    background: #00b14f !important;
}
.stSlider label { color: #c8e6c9 !important; }

details summary, details summary p,
[data-testid="stExpander"] summary span {
    color: #c8e6c9 !important;
}
[data-testid="stExpander"] {
    background: #141f14 !important;
    border: 1px solid rgba(0,177,79,0.2) !important;
    border-radius: 10px !important;
}

[data-testid="stMetricLabel"] p,
[data-testid="stMetricLabel"] span,
[data-testid="stMetricLabel"] div { color: #a5d6a7 !important; }
[data-testid="stMetricValue"] { color: #ffffff !important; }
[data-testid="stMetricDelta"] { color: #00e676 !important; }

.stCaption, .stCaption p { color: #81c784 !important; }

[data-testid="stAlert"] p,
[data-testid="stAlert"] div { color: #ffffff !important; }
.stAlert { border-radius: 12px !important; }

code {
    background: #0f1f0f !important;
    color: #00e676 !important;
    border-radius: 4px !important;
    padding: 2px 6px !important;
}
.stCodeBlock { background: #0f1f0f !important; }
.stCodeBlock code { color: #a5d6a7 !important; }

.stCheckbox label span { color: #e8f5e9 !important; }
.stCheckbox > label > div[data-testid="stMarkdownContainer"] p { color: #e8f5e9 !important; }

.stRadio label span { color: #e8f5e9 !important; }
.stRadio > div > div > label { color: #e8f5e9 !important; }

.stButton > button {
    background: linear-gradient(135deg, #00b14f, #007a35) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    font-size: 15px !important;
    padding: 12px 24px !important;
    width: 100% !important;
    transition: all 0.2s ease !important;
    box-shadow: 0 4px 15px rgba(0,177,79,0.3) !important;
}
.stButton > button:hover {
    background: linear-gradient(135deg, #00d65e, #00913d) !important;
    box-shadow: 0 6px 20px rgba(0,177,79,0.5) !important;
    transform: translateY(-1px) !important;
}
.stButton > button:active { transform: translateY(0) !important; }

.stTabs [data-baseweb=tab-list] {
    background: #111611 !important;
    border-radius: 12px !important;
    padding: 5px !important;
    gap: 3px !important;
    border: 1px solid rgba(0,177,79,0.1) !important;
}
.stTabs [data-baseweb=tab] {
    background: transparent !important;
    border-radius: 9px !important;
    color: #81c784 !important;
    font-weight: 500 !important;
    transition: all 0.2s ease !important;
}
.stTabs [aria-selected=true] {
    background: rgba(0,177,79,0.18) !important;
    color: #00e676 !important;
    font-weight: 700 !important;
    box-shadow: 0 2px 8px rgba(0,177,79,0.2) !important;
}
.stTabs [data-baseweb=tab] p { color: inherit !important; }

section[data-testid=stSidebar] {
    background: #0d150d !important;
    border-right: 1px solid rgba(0,177,79,0.15) !important;
}
section[data-testid=stSidebar] * { color: #e8f5e9 !important; }
section[data-testid=stSidebar] .stSelectbox > div > div {
    background: #1a271a !important;
    color: #e8f5e9 !important;
}

.stChatMessage {
    background: #141f14 !important;
    border: 1px solid rgba(0,177,79,0.2) !important;
    border-radius: 14px !important;
}
.stChatMessage p, .stChatMessage span { color: #e8f5e9 !important; }

.stProgress > div > div > div {
    background: linear-gradient(90deg, #00b14f, #00e676) !important;
}

hr { border-color: rgba(0,177,79,0.15) !important; }

::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: #0d150d; }
::-webkit-scrollbar-thumb { background: #00b14f55; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #00b14f; }

.driver-card {
    background: linear-gradient(135deg, #1a2e1a, #0f1f0f);
    border: 1px solid rgba(0,177,79,0.4);
    border-radius: 16px;
    padding: 20px;
    margin: 12px 0;
}
.price-card {
    background: linear-gradient(135deg, #1a2e1a, #0f1f0f);
    border: 1px solid rgba(0,177,79,0.3);
    border-radius: 16px;
    padding: 20px;
    margin: 12px 0;
}

@keyframes radar-ring {
    0%   { transform: scale(0.3); opacity: 1; }
    100% { transform: scale(2.5); opacity: 0; }
}
.radar-wrap {
    position: relative; width: 120px; height: 120px;
    display: flex; align-items: center; justify-content: center; margin: 0 auto;
}
.radar-ring {
    position: absolute; width: 120px; height: 120px;
    border-radius: 50%; border: 2px solid #00b14f;
    animation: radar-ring 2s ease-out infinite;
}
.radar-ring:nth-child(2) { animation-delay: 0.6s; }
.radar-ring:nth-child(3) { animation-delay: 1.2s; }
.radar-center {
    width: 48px; height: 48px;
    background: #00b14f; border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 22px; z-index: 10; box-shadow: 0 0 20px rgba(0,177,79,0.6);
}

@keyframes pulse-green {
    0%, 100% { box-shadow: 0 0 0 0 rgba(0,177,79,0.5); }
    50%       { box-shadow: 0 0 0 8px rgba(0,177,79,0); }
}
.live-badge {
    display: inline-block;
    background: #00b14f; color: white;
    padding: 3px 10px; border-radius: 20px;
    font-size: 12px; font-weight: 700;
    animation: pulse-green 1.8s infinite;
}

@keyframes shimmer {
    0%   { background-position: -400px 0; }
    100% { background-position: 400px 0; }
}
.shimmer {
    background: linear-gradient(90deg, #1a2a1a 25%, #243224 50%, #1a2a1a 75%);
    background-size: 400px 100%;
    animation: shimmer 1.5s infinite;
    border-radius: 8px; height: 16px; margin: 6px 0;
}

.promo-badge {
    display: inline-block;
    background: linear-gradient(135deg, #ff6b35, #ff4500);
    color: white !important; font-weight: 800;
    padding: 4px 12px; border-radius: 20px; font-size: 13px;
    letter-spacing: 0.5px;
}

.weather-card {
    background: linear-gradient(135deg, #0d2818, #1a3a1a);
    border: 1px solid rgba(0,177,79,0.25);
    border-radius: 14px; padding: 16px 20px;
    margin: 10px 0;
}

.tip-card {
    background: rgba(0,177,79,0.08);
    border-left: 3px solid #00b14f;
    border-radius: 0 10px 10px 0;
    padding: 12px 16px; margin: 8px 0;
    color: #c8e6c9 !important;
    font-size: 14px;
}

.stat-pill {
    background: linear-gradient(135deg, #1a2e1a, #0f1f0f);
    border: 1px solid rgba(0,177,79,0.3);
    border-radius: 12px; padding: 14px 18px;
    text-align: center;
}

.feature-card {
    background: linear-gradient(135deg, #1a2a1a, #0f1a0f);
    border: 1px solid rgba(0,177,79,0.25);
    border-radius: 14px;
    padding: 18px;
    margin: 8px 0;
    transition: all 0.2s ease;
}
.feature-card:hover {
    border-color: rgba(0,177,79,0.6);
    box-shadow: 0 4px 20px rgba(0,177,79,0.15);
}

@keyframes blink {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.5; }
}
.countdown { animation: blink 1s infinite; }

.surge-rain {
    background: rgba(30,100,200,0.15);
    border: 1px solid rgba(100,180,255,0.5);
    border-radius: 10px; padding: 10px 14px;
    margin: 6px 0; text-align: center;
}
.surge-traffic {
    background: rgba(255,140,0,0.15);
    border: 1px solid rgba(255,165,0,0.5);
    border-radius: 10px; padding: 10px 14px;
    margin: 6px 0; text-align: center;
}
.surge-peak {
    background: rgba(255,60,60,0.12);
    border: 1px solid rgba(255,80,80,0.45);
    border-radius: 10px; padding: 10px 14px;
    margin: 6px 0; text-align: center;
}

/* ── NEW: Spin wheel ── */
@keyframes spin-wheel {
    0%   { transform: rotate(0deg); }
    100% { transform: rotate(var(--spin-deg, 1440deg)); }
}
.wheel-spinning {
    animation: spin-wheel 3s cubic-bezier(0.17,0.67,0.12,1) forwards;
}

/* ── NEW: Pet badge ── */
.pet-badge {
    display: inline-block;
    background: linear-gradient(135deg, #ff9a9e, #fecfef);
    color: #7b2d4a !important;
    font-weight: 800; padding: 4px 14px;
    border-radius: 20px; font-size: 13px;
}

/* ── NEW: Challenge card ── */
.challenge-card {
    background: linear-gradient(135deg, #1a1a2e, #16213e);
    border: 1px solid rgba(100,100,255,0.3);
    border-radius: 14px; padding: 16px 18px; margin: 8px 0;
}
.challenge-done {
    background: linear-gradient(135deg, #0d2818, #1a3a1a);
    border: 1px solid rgba(0,177,79,0.5);
}

/* ── NEW: Wallet chart bar ── */
.wallet-bar {
    background: linear-gradient(90deg, #00b14f, #00e676);
    border-radius: 4px;
    transition: width 0.6s ease;
}
</style>
""", unsafe_allow_html=True)

ANTHROPIC_KEY = st.secrets.get("ANTHROPIC_API_KEY", "")

VEHICLES = {
    "🚲 GrabBike":       {"base": 12000, "per_km": 11000, "per_min": 400,  "emoji": "🚲", "seats": 1, "eco": True,  "pet_ok": False},
    "🚗 GrabCar 4 chỗ":  {"base": 25000, "per_km": 17000, "per_min": 700,  "emoji": "🚗", "seats": 4, "eco": False, "pet_ok": True},
    "🚙 GrabCar 7 chỗ":  {"base": 35000, "per_km": 22000, "per_min": 900,  "emoji": "🚙", "seats": 7, "eco": False, "pet_ok": True},
    "🛺 GrabTaxi":       {"base": 22000, "per_km": 16000, "per_min": 650,  "emoji": "🛺", "seats": 4, "eco": False, "pet_ok": True},
    "⚡ GrabElectric":   {"base": 20000, "per_km": 15000, "per_min": 600,  "emoji": "⚡", "seats": 4, "eco": True,  "pet_ok": True},
    "🚌 Bus Công cộng":  {"base": 7000,  "per_km": 3000,  "per_min": 150,  "emoji": "🚌", "seats": 50,"eco": True,  "pet_ok": False},
}
PROMOS  = {"GRAB20": 0.20, "NEWUSER": 0.15, "HCMC10": 0.10, "ECO15": 0.15, "FLASH30": 0.30, "PET10": 0.10}
DRIVERS = ["Nguyễn Văn Minh","Trần Thị Lan","Lê Văn Huy","Phạm Quốc Phúc","Võ Thị Thảo"]
PLATES  = ["51F-123.45","51G-678.90","51H-246.80","51K-135.79","51L-987.65"]
RATINGS = ["4.9","4.8","5.0","4.7","4.9"]
TRIPS   = ["1,247","892","2,103","567","1,456"]
PET_DRIVERS = ["Nguyễn Văn Minh 🐾", "Trần Thị Lan 🐾", "Phạm Quốc Phúc 🐾"]

SURGE_PEAK    = {"multiplier": 1.4, "label": "Giờ cao điểm", "icon": "⚡", "color": "#ff5555"}
SURGE_RAIN    = {"multiplier": 1.3, "label": "Thời tiết xấu / Mưa lớn", "icon": "🌧️", "color": "#64b5f6"}
SURGE_TRAFFIC = {"multiplier": 1.5, "label": "Kẹt xe nghiêm trọng", "icon": "🚦", "color": "#ffa040"}

TIPS = [
    "🌧️ Trời hay mưa chiều nay — đặt xe trước để tránh giá tăng cao điểm!",
    "💡 Dùng GrabBike tiết kiệm hơn 40% so với GrabCar cho quãng dưới 5km.",
    "🎯 Đặt xe trước 30 phút tránh giá surge vào 17h–19h.",
    "⭐ Đánh giá tài xế 5 sao để nhận điểm GrabRewards bonus!",
    "🔋 Pin thấp? Yêu cầu tài xế cho mượn cáp sạc trong app Chat.",
    "⚡ GrabElectric mới! Giá rẻ hơn 10% & thân thiện môi trường.",
    "🎁 Flash sale mỗi thứ 6: dùng FLASH30 giảm 30%!",
    "🐾 Có thú cưng? Bật GrabPet Mode để tìm tài xế thân thiện với động vật!",
]

LOCATIONS = {
    "quận 1": (10.7769,106.7009),"quận 2": (10.7872,106.7515),
    "quận 3": (10.7756,106.6882),"quận 4": (10.7580,106.7037),
    "quận 5": (10.7553,106.6624),"quận 6": (10.7484,106.6355),
    "quận 7": (10.7323,106.7226),"quận 8": (10.7232,106.6680),
    "quận 9": (10.8412,106.7800),"quận 10": (10.7740,106.6680),
    "quận 11": (10.7626,106.6516),"quận 12": (10.8629,106.6600),
    "bình thạnh": (10.8081,106.7084),"tân bình": (10.8014,106.6520),
    "tân phú": (10.7927,106.6282),"phú nhuận": (10.7993,106.6800),
    "gò vấp": (10.8384,106.6652),"bình tân": (10.7657,106.6075),
    "thủ đức": (10.8526,106.7516),"sân bay": (10.8188,106.6520),
    "tân sơn nhất": (10.8188,106.6520),"bến thành": (10.7725,106.6980),
    "landmark 81": (10.7950,106.7218),"bitexco": (10.7717,106.7040),
    "đại học quốc gia": (10.8800,106.8050),"suối tiên": (10.8520,106.7930),
    "crescent mall": (10.7290,106.7210),"vincom": (10.7792,106.7030),
}

FLASH_DEALS = [
    {"title": "GrabBike Flash", "discount": "35%", "code": "BIKE35", "expires": "17:00", "icon": "🚲", "left": random.randint(3,15)},
    {"title": "Car Weekend",    "discount": "20%", "code": "GRAB20", "expires": "23:59", "icon": "🚗", "left": random.randint(10,30)},
    {"title": "Electric Debut","discount": "40%", "code": "ECO15",  "expires": "18:30", "icon": "⚡", "left": random.randint(1,8)},
    {"title": "GrabPet Special","discount": "10%", "code": "PET10",  "expires": "20:00", "icon": "🐾", "left": random.randint(5,20)},
]

# ── NEW: Spin wheel prizes ──
SPIN_PRIZES = [
    {"label": "GRAB20",  "desc": "Giảm 20%",        "color": "#00b14f", "prob": 0.20},
    {"label": "5K",      "desc": "Cashback 5,000₫",  "color": "#ff9800", "prob": 0.25},
    {"label": "HCMC10",  "desc": "Giảm 10%",         "color": "#2196f3", "prob": 0.20},
    {"label": "2X PTS",  "desc": "Nhân đôi điểm",    "color": "#9c27b0", "prob": 0.15},
    {"label": "FLASH30", "desc": "Giảm 30%",          "color": "#f44336", "prob": 0.08},
    {"label": "ECO15",   "desc": "Eco -15%",          "color": "#4caf50", "prob": 0.07},
    {"label": "10K",     "desc": "Cashback 10,000₫",  "color": "#ff5722", "prob": 0.04},
    {"label": "JACKPOT", "desc": "Giảm 50%!!!",       "color": "#ffd700", "prob": 0.01},
]

# ── NEW: Daily challenges ──
DAILY_CHALLENGES = [
    {"id": "c1", "icon": "🚲", "title": "Đặt GrabBike",        "desc": "Hoàn thành 1 chuyến GrabBike hôm nay",    "pts": 30,  "xp": 50},
    {"id": "c2", "icon": "⚡", "title": "Đi xe điện",           "desc": "Thử 1 chuyến GrabElectric",              "pts": 50,  "xp": 80},
    {"id": "c3", "icon": "⭐", "title": "Đánh giá 5 sao",       "desc": "Cho tài xế 5 sao sau chuyến đi",         "pts": 20,  "xp": 30},
    {"id": "c4", "icon": "🏷️", "title": "Dùng mã giảm giá",    "desc": "Áp dụng bất kỳ mã promo nào",            "pts": 15,  "xp": 25},
    {"id": "c5", "icon": "👫", "title": "Mời bạn đi chung",     "desc": "Tạo phòng Grab Together với 1 người",    "pts": 40,  "xp": 60},
    {"id": "c6", "icon": "🌱", "title": "Hành trình xanh",      "desc": "Đặt 2 chuyến eco (Bike hoặc Electric)",  "pts": 60,  "xp": 100},
]

# ── NEW: Surge price forecast per hour (base multipliers) ──
def gen_surge_forecast():
    base = [0.9,0.85,0.8,0.8,0.85,1.0,1.3,1.5,1.4,1.1,1.0,1.0,
            1.0,1.0,1.0,1.1,1.2,1.5,1.6,1.4,1.2,1.1,1.0,0.95]
    return [round(b + random.uniform(-0.05,0.05),2) for b in base]

def get_coords(address):
    if address.startswith("GPS ("):
        try:
            inner = address[5:-1]; lat, lng = map(float, inner.split(",")); return (lat, lng)
        except: pass
    addr = address.lower().strip()
    for k, v in LOCATIONS.items():
        if k in addr: return v
    try:
        r = requests.get("https://nominatim.openstreetmap.org/search",
            params={"q": address+", TP. Hồ Chí Minh","format":"json","limit":1},
            headers={"User-Agent":"GrabVN-App/1.0"},timeout=4)
        results = r.json()
        if results: return (float(results[0]["lat"]),float(results[0]["lon"]))
    except: pass
    return (10.7769+random.uniform(-0.03,0.03), 106.7009+random.uniform(-0.03,0.03))

def get_osrm_route(p1, p2):
    try:
        url = (f"http://router.project-osrm.org/route/v1/driving/"
               f"{p1[1]},{p1[0]};{p2[1]},{p2[0]}?overview=full&geometries=geojson")
        r = requests.get(url, timeout=5); data = r.json()
        coords = data["routes"][0]["geometry"]["coordinates"]
        return [[c[1],c[0]] for c in coords]
    except: return [list(p1),list(p2)]

def compute_total_surge(surge_rain_on, surge_traffic_on, surge_peak_on):
    total = 1.0; active = []
    if surge_peak_on:    total *= SURGE_PEAK["multiplier"];    active.append(SURGE_PEAK)
    if surge_rain_on:    total *= SURGE_RAIN["multiplier"];    active.append(SURGE_RAIN)
    if surge_traffic_on: total *= SURGE_TRAFFIC["multiplier"]; active.append(SURGE_TRAFFIC)
    return round(total,2), active

# ── Session defaults ──
now_hour = datetime.now().hour
default_peak    = 7 <= now_hour <= 9 or 17 <= now_hour <= 20
default_rain    = random.random() < 0.4
default_traffic = random.random() < 0.35

for k, v in {
    "messages":           [{"role":"assistant","content":"Xin chào LeAnh! 👋 Mình là GrabBot — trợ lý AI của Grab. Bạn cần hỗ trợ gì hôm nay?"}],
    "history":            [],
    "current_ride":       None,
    "balance":            248000,
    "trip_count":         42,
    "quote":              None,
    "gps_lat":            None,
    "gps_lng":            None,
    "gps_label":          None,
    "daily_tip_idx":      random.randint(0,len(TIPS)-1),
    "saved_places":       [
        {"name":"🏠 Nhà","address":"Quận 7, TP.HCM","icon":"🏠"},
        {"name":"🏢 Công ty","address":"Landmark 81, Bình Thạnh","icon":"🏢"},
        {"name":"🏥 Bệnh viện Chợ Rẫy","address":"Quận 5, TP.HCM","icon":"🏥"},
    ],
    "sos_triggered":      False,
    "rating_submitted":   False,
    "last_rating":        0,
    "carbon_saved":       0.0,
    "loyalty_points":     3450,
    "streak_days":        7,
    "surge_peak":         default_peak,
    "surge_rain":         default_rain,
    "surge_traffic":      default_traffic,
    # ── NEW state ──
    "pet_mode":           False,
    "pet_type":           "🐶 Chó",
    "pet_name":           "",
    "pet_weight":         "Nhỏ (<5kg)",
    "spin_available":     2,
    "spin_history":       [],
    "challenges_done":    set(),
    "challenge_pts_today": 0,
    "together_room":      None,
    "together_members":   [],
    "wallet_txns":        [
        {"date":"01/05","desc":"GrabBike Q1→Q3","amt":-45000,"cat":"🚲"},
        {"date":"30/04","desc":"GrabCar Sân Bay","amt":-185000,"cat":"🚗"},
        {"date":"29/04","desc":"GrabPay top-up","amt":200000,"cat":"💚"},
        {"date":"28/04","desc":"GrabElectric Q7","amt":-62000,"cat":"⚡"},
        {"date":"27/04","desc":"Flash Sale GrabBike","amt":-28000,"cat":"🚲"},
        {"date":"26/04","desc":"GrabCar 7 chỗ","amt":-210000,"cat":"🚙"},
        {"date":"25/04","desc":"GrabPay top-up","amt":300000,"cat":"💚"},
        {"date":"24/04","desc":"GrabPet + GrabCar","amt":-95000,"cat":"🐾"},
    ],
    "surge_forecast":     gen_surge_forecast(),
}.items():
    if k not in st.session_state: st.session_state[k] = v

# ── GPS from query params ──
qp = st.query_params
if "lat" in qp and "lng" in qp:
    try:
        st.session_state.gps_lat   = float(qp["lat"])
        st.session_state.gps_lng   = float(qp["lng"])
        st.session_state.gps_label = "📍 Vị trí hiện tại của bạn"
        st.query_params.clear()
    except: pass

# ──────────────────────────────────────────────────
# SIDEBAR
# ──────────────────────────────────────────────────
with st.sidebar:
    pet_indicator = " 🐾" if st.session_state.pet_mode else ""
    st.markdown(f"""
    <div style="text-align:center; padding:10px 0 20px;">
        <div style="width:64px; height:64px; background:linear-gradient(135deg,#00b14f,#005c28);
             border-radius:50%; display:flex; align-items:center; justify-content:center;
             font-size:28px; margin:0 auto 10px; box-shadow:0 0 20px rgba(0,177,79,0.4);">🧑</div>
        <div style="font-size:18px; font-weight:800; color:#ffffff;">LeAnh{pet_indicator}</div>
        <div style="font-size:12px; color:#81c784; margin-top:2px;">⭐ GrabRewards Gold</div>
        <div style="margin-top:8px; background:rgba(0,177,79,0.1); border-radius:20px;
             padding:4px 14px; font-size:12px; color:#00e676; display:inline-block;">
            🔥 {st.session_state.streak_days}-ngày liên tiếp
        </div>
    </div>
    """, unsafe_allow_html=True)

    col_a, col_b = st.columns(2)
    col_a.metric("💚 GrabPay", f"{st.session_state.balance:,}₫")
    col_b.metric("🛵 Chuyến", st.session_state.trip_count)

    st.markdown(f"""
    <div style="background:rgba(0,200,83,0.08); border:1px solid rgba(0,200,83,0.3);
         border-radius:10px; padding:10px 14px; margin:8px 0; text-align:center;">
        <div style="color:#69f0ae; font-size:11px; text-transform:uppercase; letter-spacing:1px;">🌱 Carbon đã tiết kiệm</div>
        <div style="color:#00e676; font-size:20px; font-weight:800; margin-top:2px;">{st.session_state.carbon_saved:.2f} kg CO₂</div>
    </div>
    """, unsafe_allow_html=True)

    # ── NEW: spin wheel counter in sidebar ──
    if st.session_state.spin_available > 0:
        st.markdown(f"""
        <div style="background:rgba(255,165,0,0.12); border:1px solid rgba(255,165,0,0.4);
             border-radius:10px; padding:8px 14px; margin:6px 0; text-align:center;">
            <div style="color:#ffd54f; font-weight:700; font-size:13px;">🎰 Bạn có {st.session_state.spin_available} lượt quay!</div>
            <div style="color:#ffb74d; font-size:11px;">Vào tab Vòng Quay để nhận thưởng</div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")
    tip = TIPS[st.session_state.daily_tip_idx]
    st.markdown(f'<div class="tip-card">{tip}</div>', unsafe_allow_html=True)
    st.markdown("---")

    vehicle = st.selectbox("🚗 Phương tiện", list(VEHICLES.keys()), key="vehicle_select")

    # ── NEW: Pet mode toggle in sidebar ──
    st.markdown("---")
    pet_on = st.checkbox("🐾 Bật GrabPet Mode", value=st.session_state.pet_mode, key="cb_pet_sidebar")
    st.session_state.pet_mode = pet_on
    if pet_on:
        st.markdown('<span class="pet-badge">🐾 GrabPet Active</span>', unsafe_allow_html=True)
        st.session_state.pet_type   = st.selectbox("Loài thú cưng", ["🐶 Chó","🐱 Mèo","🐰 Thỏ","🐦 Chim","🐹 Chuột hamster"], key="pet_type_sb")
        st.session_state.pet_weight = st.selectbox("Cân nặng",     ["Nhỏ (<5kg)","Vừa (5-15kg)","Lớn (>15kg)"], key="pet_wt_sb")

    st.markdown("---")
    st.markdown("**⚡ Điều kiện đường hiện tại:**")
    surge_peak_on    = st.checkbox("⚡ Giờ cao điểm (+40%)",  value=st.session_state.surge_peak,    key="cb_peak")
    surge_rain_on    = st.checkbox("🌧️ Mưa lớn / Bão (+30%)", value=st.session_state.surge_rain,    key="cb_rain")
    surge_traffic_on = st.checkbox("🚦 Kẹt xe nặng (+50%)",   value=st.session_state.surge_traffic, key="cb_traffic")
    st.session_state.surge_peak    = surge_peak_on
    st.session_state.surge_rain    = surge_rain_on
    st.session_state.surge_traffic = surge_traffic_on

    total_surge, active_surges = compute_total_surge(surge_rain_on, surge_traffic_on, surge_peak_on)
    if active_surges:
        labels = " + ".join(f"{s['icon']} {s['label']}" for s in active_surges)
        border_c = active_surges[-1]["color"]
        st.markdown(f"""
        <div style="background:rgba(255,100,50,0.1); border:1px solid {border_c};
             border-radius:10px; padding:10px 14px; text-align:center; margin-top:6px;">
            <div style="color:{border_c}; font-weight:800; font-size:14px;">GIÁ ĐANG TĂNG</div>
            <div style="color:#ffcba4; font-size:12px; margin-top:3px;">{labels}</div>
            <div style="color:#ffffff; font-size:20px; font-weight:800; margin-top:4px;">×{total_surge}</div>
        </div>
        """, unsafe_allow_html=True)
    else:
        st.markdown("""
        <div style="background:rgba(0,177,79,0.1); border:1px solid rgba(0,177,79,0.4);
             border-radius:10px; padding:10px 14px; text-align:center; margin-top:6px;">
            <div style="color:#00e676; font-weight:700; font-size:13px;">✅ Giá bình thường</div>
            <div style="color:#81c784; font-size:12px; margin-top:2px;">Không có phụ phí</div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("**⚡ Flash Deals hôm nay:**")
    for deal in FLASH_DEALS:
        st.markdown(f"""
        <div style="background:#1a1a0a; border:1px solid rgba(255,165,0,0.3);
             border-radius:8px; padding:8px 10px; margin:4px 0;
             display:flex; justify-content:space-between; align-items:center;">
            <div>
                <span style="font-size:14px;">{deal['icon']}</span>
                <span style="color:#ffd54f; font-size:12px; font-weight:700; margin-left:4px;">{deal['title']}</span>
                <div style="color:#ff9800; font-size:11px;">Còn {deal['left']} lượt · hết {deal['expires']}</div>
            </div>
            <span style="background:#ff6b35; color:white; padding:3px 8px;
                border-radius:12px; font-size:12px; font-weight:800;">-{deal['discount']}</span>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")
    if surge_rain_on:
        weather_icon, weather_temp, weather_desc, weather_extra = "⛈️","29°C","Mưa lớn — cẩn thận ngập lụt!","💧 Độ ẩm 95% · 💨 Gió NE 30km/h"
    elif surge_traffic_on:
        weather_icon, weather_temp, weather_desc, weather_extra = "🌩️","31°C","Có mây, tắc đường nhiều tuyến","💧 Độ ẩm 82% · 🚦 Kẹt xe: Q.1, Q.3, Q.Bình Thạnh"
    else:
        weather_icon, weather_temp, weather_desc, weather_extra = "⛅","32°C","Có mây, khả năng mưa chiều","💧 Độ ẩm 78% · 💨 Gió NE 15km/h"
    st.markdown(f"""
    <div class="weather-card">
        <div style="display:flex; align-items:center; gap:10px;">
            <span style="font-size:28px;">{weather_icon}</span>
            <div>
                <div style="color:#ffffff; font-weight:700;">TP.HCM • {weather_temp}</div>
                <div style="color:#81c784; font-size:12px;">{weather_desc}</div>
                <div style="color:#a5d6a7; font-size:11px; margin-top:3px;">{weather_extra}</div>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

# ── Tabs ──
tab1, tab2, tab3, tab4, tab5, tab6, tab7, tab8, tab9 = st.tabs([
    "🛵 Đặt Chuyến", "📍 Theo Dõi", "📜 Lịch Sử",
    "💬 Hỗ Trợ AI", "🎁 Ưu Đãi",
    "🎰 Vòng Quay",   "🎯 Thử Thách",
    "👫 Đi Chung",    "💳 Ví & Chi Tiêu"
])

# ──────────────────────────────────────────────────
# TAB 1 — ĐẶT CHUYẾN
# ──────────────────────────────────────────────────
with tab1:
    st.markdown("""
    <div style="background:linear-gradient(135deg,#0d2818,#1a3a1a);
         border:1px solid rgba(0,177,79,0.3); border-radius:16px;
         padding:20px 24px; margin-bottom:20px; display:flex; align-items:center; gap:16px;">
        <span style="font-size:40px;">🚕</span>
        <div>
            <div style="font-size:20px; font-weight:800; color:#ffffff;">Đặt xe ngay</div>
            <div style="color:#81c784; font-size:14px;">
                Hơn 500 tài xế đang hoạt động gần bạn
                <span class="live-badge" style="margin-left:8px;">LIVE</span>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    # ── NEW: GrabPet banner ──
    if st.session_state.pet_mode:
        veh_info = VEHICLES.get(vehicle, {})
        pet_ok = veh_info.get("pet_ok", False)
        if pet_ok:
            st.markdown(f"""
            <div style="background:linear-gradient(135deg,#2a1a2a,#3a1a3a);
                 border:2px solid rgba(255,154,158,0.6); border-radius:14px;
                 padding:14px 20px; margin-bottom:16px; display:flex; align-items:center; gap:14px;">
                <span style="font-size:32px;">{st.session_state.pet_type.split()[0]}</span>
                <div>
                    <div style="color:#ffc8d0; font-weight:800; font-size:15px;">🐾 GrabPet Mode đang bật</div>
                    <div style="color:#f8a8b8; font-size:13px;">
                        Tìm tài xế thân thiện thú cưng · {st.session_state.pet_type} · {st.session_state.pet_weight}
                    </div>
                    <div style="color:#ff9a9e; font-size:12px; margin-top:3px;">
                        Dùng mã <b>PET10</b> để giảm thêm 10% cho chuyến này 🐾
                    </div>
                </div>
            </div>
            """, unsafe_allow_html=True)
        else:
            st.warning(f"⚠️ **{vehicle}** không hỗ trợ GrabPet. Vui lòng chọn GrabCar, GrabTaxi hoặc GrabElectric để mang thú cưng.")

    total_surge, active_surges = compute_total_surge(
        st.session_state.surge_rain, st.session_state.surge_traffic, st.session_state.surge_peak)
    if active_surges:
        surge_lines = []
        for s in active_surges:
            cls = {"⚡":"surge-peak","🌧️":"surge-rain","🚦":"surge-traffic"}[s["icon"]]
            surge_lines.append(f'<div class="{cls}" style="flex:1;">'
                f'<span style="font-size:18px;">{s["icon"]}</span>'
                f'<span style="color:#fff; font-weight:700; font-size:13px; margin-left:6px;">{s["label"]}</span>'
                f'<span style="color:#ffd; font-size:12px; margin-left:6px;">×{s["multiplier"]}</span></div>')
        st.markdown(f"""
        <div style="display:flex; gap:8px; margin-bottom:16px; flex-wrap:wrap;">{''.join(surge_lines)}</div>
        <div style="background:rgba(255,80,50,0.1); border:1px solid rgba(255,100,60,0.5);
             border-radius:12px; padding:10px 16px; margin-bottom:16px; text-align:center;">
            <span style="color:#ff9a6c; font-weight:800;">⚠️ Tổng hệ số giá: ×{total_surge}</span>
        </div>""", unsafe_allow_html=True)

    st.markdown("**⭐ Địa điểm thường đến:**")
    fav_cols = st.columns(len(st.session_state.saved_places))
    for i, (col, place) in enumerate(zip(fav_cols, st.session_state.saved_places)):
        if col.button(f"{place['icon']} {place['name'].replace(place['icon']+' ','')}", key=f"fav_{i}", use_container_width=True):
            st.session_state["dropoff_prefill"] = place["address"]; st.rerun()

    st.subheader("📍 Thông tin chuyến đi")

    gps_html = """
    <script>
    function useMyLocation() {
        var btn=document.getElementById('gpsbtn'); var status=document.getElementById('gpsstatus');
        btn.disabled=true; btn.innerText='⏳ Đang lấy GPS...';
        if(!navigator.geolocation){status.innerText='❌ Trình duyệt không hỗ trợ GPS';btn.disabled=false;btn.innerText='📍 Dùng vị trí hiện tại';return;}
        navigator.geolocation.getCurrentPosition(function(pos){
            var lat=pos.coords.latitude.toFixed(6); var lng=pos.coords.longitude.toFixed(6);
            status.innerText='✅ '+lat+', '+lng+' — đang cập nhật...';
            var url=new URL(window.parent.location.href);
            url.searchParams.set('lat',lat); url.searchParams.set('lng',lng);
            window.parent.location.href=url.toString();
        },function(err){status.innerText='❌ '+err.message;btn.disabled=false;btn.innerText='📍 Dùng vị trí hiện tại';},{enableHighAccuracy:true,timeout:10000});
    }
    </script>
    <div style="background:#1a241a; padding:12px 16px; border-radius:12px;
         border:1px solid rgba(0,177,79,0.3); margin-bottom:4px; display:flex; align-items:center; gap:14px; flex-wrap:wrap;">
        <button id="gpsbtn" onclick="useMyLocation()" style="background:linear-gradient(135deg,#00b14f,#007a35);color:white;
            border:none;border-radius:8px;padding:9px 18px;font-weight:700;font-size:14px;cursor:pointer;white-space:nowrap;box-shadow:0 3px 10px rgba(0,177,79,0.3);">
            📍 Dùng vị trí hiện tại làm điểm đón</button>
        <span id="gpsstatus" style="color:#a5d6a7;font-size:13px;">Bấm để lấy GPS</span>
    </div>"""
    st.components.v1.html(gps_html, height=68)

    pickup_default = "Quận 1, TP.HCM"
    if st.session_state.gps_lat is not None:
        st.success(f"🛰️ Đã lấy GPS: **{st.session_state.gps_lat:.5f}, {st.session_state.gps_lng:.5f}**")
        pickup_default = f"GPS ({st.session_state.gps_lat:.5f}, {st.session_state.gps_lng:.5f})"

    dropoff_default = st.session_state.get("dropoff_prefill","")
    if dropoff_default: st.session_state["dropoff_prefill"] = ""

    col1, col2 = st.columns(2)
    with col1: pickup  = st.text_input("📍 Điểm đón", value=pickup_default, key="pickup_input")
    with col2: dropoff = st.text_input("🏁 Điểm đến", value=dropoff_default, placeholder="Bạn muốn đến đâu?", key="dropoff_input")

    col3, col4, col5 = st.columns(3)
    with col3: schedule_type = st.selectbox("⏰ Thời gian",    ["⚡ Đi ngay","📅 Đặt lịch trước"], key="schedule_select")
    with col4: payment       = st.selectbox("💳 Thanh toán",   ["💚 GrabPay","💵 Tiền mặt","💳 Thẻ ngân hàng"], key="payment_select")
    with col5: passengers    = st.selectbox("👥 Số hành khách", ["1 người","2 người","3 người","4 người"], key="pax_select")

    if schedule_type == "📅 Đặt lịch trước":
        sc1, sc2 = st.columns(2)
        with sc1: st.date_input("📅 Ngày đặt", key="sched_date")
        with sc2: st.time_input("🕐 Giờ đặt",  key="sched_time")

    col6, col7 = st.columns(2)
    with col6: promo_code = st.text_input("🏷️ Mã khuyến mãi", placeholder="VD: GRAB20, PET10", key="promo_input")
    with col7: note       = st.text_input("📝 Ghi chú cho tài xế", placeholder="VD: Đứng trước cổng màu xanh...", key="note_input")

    # ── NEW: GrabPet extra options ──
    if st.session_state.pet_mode:
        with st.expander("🐾 Thông tin thú cưng"):
            p1, p2, p3 = st.columns(3)
            with p1: st.session_state.pet_name = st.text_input("Tên thú cưng", value=st.session_state.pet_name, placeholder="VD: Mochi", key="pet_name_inp")
            with p2: st.selectbox("Tình trạng sức khỏe", ["Bình thường","Đang điều trị","Mang thai"], key="pet_health")
            with p3: st.checkbox("Có lồng/túi chứa", key="pet_cage")
            st.text_area("Ghi chú thú cưng", placeholder="VD: Mochi hay sợ tiếng còi, xin tài xế lái nhẹ nhàng...", key="pet_note", height=80)

    with st.expander("⚙️ Tùy chọn thêm"):
        opt1, opt2, opt3 = st.columns(3)
        with opt1:
            st.checkbox("🧳 Có hành lý lớn", key="has_luggage")
            st.checkbox("👶 Cần ghế trẻ em", key="child_seat")
        with opt2:
            st.checkbox("🔇 Không cần nói chuyện", key="silent_ride")
            st.checkbox("❄️ Bật máy lạnh", key="ac_on")
        with opt3:
            st.checkbox("🎵 Ưa nhạc nhẹ", key="soft_music")
            st.checkbox("♿ Hỗ trợ người khuyết tật", key="accessibility")

    if dropoff:
        st.markdown("---")
        st.markdown("**💡 So sánh giá nhanh (ước tính ~5km):**")
        cols = st.columns(len(VEHICLES))
        for i, (vname, vinfo) in enumerate(VEHICLES.items()):
            est = round((vinfo["base"]+vinfo["per_km"]*5+vinfo["per_min"]*20)*total_surge/1000)*1000
            selected_style = "border:2px solid #00b14f !important;" if vname==vehicle else ""
            eco_b = '<div style="color:#69f0ae;font-size:10px;margin-top:2px;">🌱 Eco</div>' if vinfo["eco"] else ""
            pet_b = '<div style="color:#ffc8d0;font-size:10px;margin-top:2px;">🐾 Pet OK</div>' if vinfo["pet_ok"] else ""
            cols[i].markdown(f"""
            <div style="background:#141f14;border:1px solid rgba(0,177,79,0.25);{selected_style}border-radius:10px;padding:10px;text-align:center;">
                <div style="font-size:22px;">{vinfo['emoji']}</div>
                <div style="color:#c8e6c9;font-size:10px;margin-top:3px;">{vname.split(' ',1)[1] if ' ' in vname else vname}</div>
                <div style="color:#00e676;font-weight:700;font-size:13px;margin-top:4px;">{est:,}₫</div>
                {eco_b}{pet_b}
            </div>""", unsafe_allow_html=True)

    st.markdown("---")
    if st.button("🔍 Tính Giá & Tìm Tài Xế", use_container_width=True):
        if not pickup or not dropoff:
            st.error("❌ Vui lòng nhập đủ điểm đón và điểm đến!")
            st.session_state.quote = None
        else:
            with st.spinner("⏳ Đang tính toán lộ trình..."):
                time.sleep(1)
            dist_km = round(random.uniform(2,15),1)
            dur_min = int(dist_km*3.5+random.uniform(2,8))
            if st.session_state.surge_traffic: dur_min = int(dur_min*1.6)
            elif st.session_state.surge_peak:  dur_min = int(dur_min*1.3)
            v = VEHICLES[vehicle]
            current_surge, surge_list = compute_total_surge(
                st.session_state.surge_rain, st.session_state.surge_traffic, st.session_state.surge_peak)
            # Pet surcharge
            pet_fee = 0
            if st.session_state.pet_mode and v.get("pet_ok"):
                wt = st.session_state.pet_weight
                pet_fee = 15000 if wt=="Nhỏ (<5kg)" else (25000 if wt=="Vừa (5-15kg)" else 40000)
            raw   = (v["base"]+v["per_km"]*dist_km+v["per_min"]*dur_min)*current_surge + pet_fee
            total = round(raw/1000)*1000
            discount = 0; promo_pct = PROMOS.get(promo_code.upper().strip(),0)
            promo_msg=""; promo_warn=""
            if promo_pct:
                discount = round(total*promo_pct/1000)*1000; total -= discount
                promo_msg = f"✅ Mã áp dụng: -{discount:,}₫ (tiết kiệm {int(promo_pct*100)}%!)"
            elif promo_code.strip():
                promo_warn = "⚠️ Mã không hợp lệ. Thử: GRAB20, NEWUSER, HCMC10, ECO15, FLASH30, PET10"
            co2_saved = round(dist_km*0.12,2) if v.get("eco") else 0
            st.session_state.quote = {
                "dist":dist_km,"mins":dur_min,"total":total,"discount":discount,
                "pickup":pickup,"dropoff":dropoff,"vehicle":vehicle,"v":v,
                "promo_msg":promo_msg,"promo_warn":promo_warn,
                "co2_saved":co2_saved,"note":note,"surge":current_surge,
                "surge_list":surge_list,"pet_fee":pet_fee,
                "pet_mode":st.session_state.pet_mode,
                "pet_type":st.session_state.pet_type,
            }

    q = st.session_state.quote
    if q:
        if q["promo_msg"]: st.success(q["promo_msg"])
        if q["promo_warn"]: st.warning(q["promo_warn"])
        if q["co2_saved"]>0:
            st.markdown(f"""
            <div style="background:rgba(0,200,83,0.1);border:1px solid rgba(0,200,83,0.3);border-radius:10px;padding:10px 16px;margin:8px 0;display:flex;gap:10px;align-items:center;">
                <span style="font-size:20px;">🌱</span>
                <span style="color:#69f0ae;font-size:14px;">Chuyến này tiết kiệm <b>{q['co2_saved']}kg CO₂</b>!</span>
            </div>""", unsafe_allow_html=True)
        if q.get("pet_fee",0)>0:
            st.markdown(f"""
            <div style="background:rgba(255,154,158,0.1);border:1px solid rgba(255,154,158,0.3);border-radius:10px;padding:10px 16px;margin:8px 0;display:flex;gap:10px;align-items:center;">
                <span style="font-size:20px;">{q['pet_type'].split()[0]}</span>
                <span style="color:#ffc8d0;font-size:14px;">Phụ phí GrabPet ({q['pet_type']}): <b>+{q['pet_fee']:,}₫</b></span>
            </div>""", unsafe_allow_html=True)
        c1,c2,c3 = st.columns(3)
        c1.metric("📏 Khoảng cách",f"{q['dist']} km")
        c2.metric("⏱ Thời gian",   f"~{q['mins']} phút")
        c3.metric("⚡ Surge",       f"{q['surge']}×")
        st.markdown("---")
        st.markdown(f"""
        <div style="background:linear-gradient(135deg,#0d2818,#1a3a1a);border:2px solid rgba(0,177,79,0.5);
             border-radius:16px;padding:20px 24px;text-align:center;">
            <div style="color:#a5d6a7;font-size:14px;margin-bottom:4px;">Tổng thanh toán</div>
            <div style="font-size:36px;font-weight:800;color:#00e676;">{q['total']:,} ₫</div>
            <div style="color:#81c784;font-size:14px;margin-top:6px;">{q['vehicle']} · {q['pickup']} → {q['dropoff']}</div>
        </div>""", unsafe_allow_html=True)
        st.markdown("---")
        split_count = st.slider("👥 Chia tiền cho bao nhiêu người?",1,6,1,key="split_slider")
        if split_count>1:
            per_person = round(q["total"]/split_count/1000)*1000
            st.markdown(f"""
            <div style="background:rgba(0,177,79,0.1);border:1px solid rgba(0,177,79,0.3);
                 border-radius:10px;padding:12px 16px;text-align:center;">
                <span style="color:#a5d6a7;">Mỗi người đóng: </span>
                <span style="color:#00e676;font-size:20px;font-weight:800;">{per_person:,}₫</span>
                <span style="color:#a5d6a7;"> / {split_count} người</span>
            </div>""", unsafe_allow_html=True)
        st.markdown("---")
        if st.button("✅ Xác Nhận Đặt Xe — Tìm Tài Xế Ngay", use_container_width=True, key="confirm_btn"):
            radar_ph = st.empty()
            radar_ph.markdown("""
            <div style="text-align:center;padding:30px 0;">
                <div class="radar-wrap" style="margin-bottom:16px;">
                    <div class="radar-ring"></div><div class="radar-ring"></div><div class="radar-ring"></div>
                    <div class="radar-center">🛵</div>
                </div>
                <p style="color:#00b14f;font-weight:700;font-size:16px;margin-top:60px;">Đang tìm tài xế gần bạn...</p>
            </div>""", unsafe_allow_html=True)
            time.sleep(3); radar_ph.empty()
            with st.spinner("🗺️ Đang tải tuyến đường..."):
                pickup_coords  = get_coords(q["pickup"])
                dropoff_coords = get_coords(q["dropoff"])
                route = get_osrm_route(pickup_coords, dropoff_coords)
            idx = random.randint(0,len(DRIVERS)-1)
            driver_name = (random.choice(PET_DRIVERS) if q.get("pet_mode") and q["v"].get("pet_ok") else DRIVERS[idx])
            st.session_state.current_ride = {
                "id":f"GR{random.randint(100000,999999)}","driver":driver_name,
                "plate":PLATES[idx],"rating":RATINGS[idx],"trips":TRIPS[idx],
                "vehicle":q["vehicle"],"pickup":q["pickup"],"dropoff":q["dropoff"],
                "pickup_coords":pickup_coords,"dropoff_coords":dropoff_coords,
                "route":route,"dist":q["dist"],"mins":q["mins"],"fare":q["total"],
                "status":"Đang đến","note":q.get("note",""),"eta":random.randint(3,8),
                "co2_saved":q["co2_saved"],"surge":q["surge"],"pet_mode":q.get("pet_mode",False),
            }
            st.session_state.balance -= q["total"]
            st.session_state.quote = None
            st.session_state.rating_submitted = False
            # Award spin after every completed booking
            st.session_state.spin_available += 1
            # Mark challenge c1 or c2 done
            if "GrabBike" in q["vehicle"]: st.session_state.challenges_done.add("c1")
            if "Electric"  in q["vehicle"]: st.session_state.challenges_done.add("c2")
            if promo_code.strip(): st.session_state.challenges_done.add("c4")
            st.success(f"🎉 Đặt thành công! **{driver_name}** đang đến. Bạn nhận được 🎰 1 lượt quay thưởng!")
            st.rerun()

    if st.session_state.current_ride:
        ride = st.session_state.current_ride
        pet_note = " 🐾" if ride.get("pet_mode") else ""
        st.markdown(f"""
        <div style="background:rgba(0,177,79,0.1);border:1px solid rgba(0,177,79,0.4);border-radius:12px;padding:14px 18px;margin-top:12px;display:flex;align-items:center;gap:12px;">
            <span style="font-size:22px;">🟢</span>
            <div>
                <span style="color:#00e676;font-weight:700;">Đang có chuyến với {ride['driver']}{pet_note}</span>
                <div style="color:#81c784;font-size:12px;margin-top:2px;">Xem chi tiết ở tab 📍 Theo Dõi</div>
            </div>
        </div>""", unsafe_allow_html=True)

# ──────────────────────────────────────────────────
# TAB 2 — THEO DÕI (unchanged core, minor pet note)
# ──────────────────────────────────────────────────
with tab2:
    st.subheader("📍 Theo dõi chuyến đi")
    ride = st.session_state.current_ride

    loc_html = """<script>
    function getLocation(){document.getElementById('status').innerText='⏳ Đang lấy vị trí...';
    if(navigator.geolocation){navigator.geolocation.getCurrentPosition(function(pos){
        var lat=pos.coords.latitude.toFixed(6);var lng=pos.coords.longitude.toFixed(6);
        document.getElementById('coords').innerText='📌 '+lat+', '+lng;
        document.getElementById('status').innerText='✅ Đã lấy vị trí!';
        document.getElementById('box').style.borderColor='#00b14f';},
        function(err){document.getElementById('status').innerText='❌ '+err.message;},{enableHighAccuracy:true,timeout:10000});}}
    </script>
    <div id="box" style="background:#1a241a;padding:14px;border-radius:12px;border:1px solid rgba(0,177,79,0.3);margin-bottom:10px;">
        <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
            <button onclick="getLocation()" style="background:linear-gradient(135deg,#00b14f,#007a35);color:white;border:none;border-radius:8px;padding:9px 18px;font-weight:700;font-size:14px;cursor:pointer;">📍 Lấy vị trí hiện tại</button>
            <span id="status" style="color:#a5d6a7;font-size:14px;">Bấm để lấy GPS...</span>
        </div>
        <div id="coords" style="color:#00e676;font-size:13px;margin-top:8px;font-family:monospace;"></div>
    </div>"""
    st.components.v1.html(loc_html, height=100)

    if not ride:
        st.info("🛵 Chưa có chuyến đang chạy. Đặt xe ở tab **🛵 Đặt Chuyến**.")
        if st.session_state.gps_lat is not None:
            center, zoom = [st.session_state.gps_lat,st.session_state.gps_lng], 15
        else:
            center, zoom = [10.7769,106.7009], 13
        m = folium.Map(location=center,zoom_start=zoom,tiles="OpenStreetMap")
        LocateControl(auto_start=False,position="topright").add_to(m)
        if st.session_state.gps_lat is not None:
            folium.Marker(center,popup="📍 Vị trí hiện tại",icon=folium.Icon(color="green",icon="user",prefix="fa")).add_to(m)
        else:
            folium.Marker(center,popup="TP. Hồ Chí Minh",icon=folium.Icon(color="green",icon="map-marker")).add_to(m)
        st_folium(m,width="100%",height=420)
    else:
        pet_label = f" 🐾 {st.session_state.pet_type}" if ride.get("pet_mode") else ""
        st.markdown(f"""
        <div style="display:flex;gap:12px;margin-bottom:16px;flex-wrap:wrap;">
            <div style="background:rgba(0,177,79,0.12);border:1px solid rgba(0,177,79,0.4);border-radius:12px;padding:12px 20px;text-align:center;flex:1;">
                <div style="color:#a5d6a7;font-size:12px;text-transform:uppercase;">ETA</div>
                <div style="color:#00e676;font-size:28px;font-weight:800;">{ride.get('eta',5)} phút</div>
                <div style="color:#81c784;font-size:11px;">Tài xế đang đến</div>
            </div>
            <div style="background:rgba(0,177,79,0.12);border:1px solid rgba(0,177,79,0.4);border-radius:12px;padding:12px 20px;text-align:center;flex:1;">
                <div style="color:#a5d6a7;font-size:12px;text-transform:uppercase;">Quãng đường</div>
                <div style="color:#00e676;font-size:28px;font-weight:800;">{ride['dist']} km</div>
                <div style="color:#81c784;font-size:11px;">~{ride['mins']} phút đến nơi</div>
            </div>
            <div style="background:rgba(0,177,79,0.12);border:1px solid rgba(0,177,79,0.4);border-radius:12px;padding:12px 20px;text-align:center;flex:1;">
                <div style="color:#a5d6a7;font-size:12px;text-transform:uppercase;">Tổng tiền</div>
                <div style="color:#00e676;font-size:22px;font-weight:800;">{ride['fare']:,}₫</div>
                <div style="color:#81c784;font-size:11px;">Đã trừ GrabPay</div>
            </div>
        </div>""", unsafe_allow_html=True)

        st.markdown(f"""
        <div class="driver-card">
            <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
                <div style="width:64px;height:64px;background:linear-gradient(135deg,#00b14f,#005c28);border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:28px;box-shadow:0 0 20px rgba(0,177,79,0.4);">
                    {"🐾" if ride.get("pet_mode") else "🧑‍✈️"}</div>
                <div style="flex:1;">
                    <div style="font-size:20px;font-weight:700;color:#ffffff;">{ride['driver']}{pet_label}</div>
                    <div style="color:#a5d6a7;font-size:14px;margin-top:2px;">⭐ {ride['rating']} · 🛵 {ride['trips']} chuyến</div>
                    <div style="color:#81c784;font-size:13px;margin-top:4px;">{ride['vehicle']} · <code style="background:#0f1f0f;padding:2px 8px;border-radius:4px;color:#00e676;">{ride['plate']}</code></div>
                </div>
                <div style="text-align:right;">
                    <span style="background:rgba(0,177,79,0.2);color:#00b14f;padding:6px 14px;border-radius:20px;font-size:13px;font-weight:700;">🟢 {ride['status']}</span>
                    <div style="margin-top:8px;color:#81c784;font-size:12px;">Mã: <code style="color:#00e676;">{ride['id']}</code></div>
                </div>
            </div>
            <div style="margin-top:14px;padding-top:14px;border-top:1px solid rgba(0,177,79,0.2);color:#c8e6c9;font-size:14px;line-height:1.8;">
                📍 <b>{ride['pickup']}</b> → 🏁 <b>{ride['dropoff']}</b>
                {f'<br>📝 {ride["note"]}' if ride.get("note") else ""}
                {f'<br>🌱 Tiết kiệm <b>{ride["co2_saved"]}kg CO₂</b>' if ride.get("co2_saved",0)>0 else ""}
            </div>
        </div>""", unsafe_allow_html=True)

        p1,p2 = ride["pickup_coords"],ride["dropoff_coords"]
        route_coords = ride.get("route",[p1,p2])
        mid = ((p1[0]+p2[0])/2,(p1[1]+p2[1])/2)
        m = folium.Map(location=mid,zoom_start=13,tiles="OpenStreetMap")
        LocateControl(auto_start=False,position="topright").add_to(m)
        AntPath(route_coords,color="#00b14f",weight=5,opacity=0.85,delay=800).add_to(m)
        folium.Marker(p1,popup=f"📍 {ride['pickup']}",icon=folium.Icon(color="green",icon="play")).add_to(m)
        folium.Marker(p2,popup=f"🏁 {ride['dropoff']}",icon=folium.Icon(color="red",icon="flag")).add_to(m)
        driver_pos = route_coords[max(1,len(route_coords)//3)] if len(route_coords)>=3 else [(p1[0]+p2[0])/2+0.003,(p1[1]+p2[1])/2+0.003]
        folium.Marker(driver_pos,popup=f"🛵 {ride['driver']}",icon=folium.Icon(color="blue",icon="car",prefix="fa")).add_to(m)
        if st.session_state.gps_lat is not None:
            folium.Marker([st.session_state.gps_lat,st.session_state.gps_lng],popup="📱 Bạn",icon=folium.Icon(color="purple",icon="user",prefix="fa")).add_to(m)
        st_folium(m,width="100%",height=460)
        st.markdown("---")

        sos_col,a_col,b_col,c_col = st.columns([1,1,1,1])
        with sos_col:
            if st.button("🆘 SOS",use_container_width=True,key="sos_btn"): st.session_state.sos_triggered=True
        with a_col:
            if st.button("🎉 Hoàn thành chuyến",use_container_width=True):
                st.session_state.history.insert(0,{**ride,"date":datetime.now().strftime("%d/%m/%Y %H:%M"),"rating_given":5})
                st.session_state.trip_count += 1
                st.session_state.carbon_saved += ride.get("co2_saved",0)
                st.session_state.loyalty_points += 50
                st.session_state.spin_available += 1  # bonus spin on completion
                st.session_state.challenges_done.add("c3")  # assume rated 5 stars
                st.session_state.current_ride = None
                st.session_state.rating_submitted = False
                st.balloons()
                st.success("🎉 Hoàn thành! +50 điểm & 🎰 1 lượt quay thêm!")
                st.rerun()
        with b_col:
            if st.button("💬 Chat tài xế",use_container_width=True): st.info("📱 Đang mở chat... (demo)")
        with c_col:
            if st.button("✕ Hủy chuyến",use_container_width=True):
                st.session_state.balance += ride["fare"]
                st.session_state.current_ride = None
                st.warning("Đã hủy chuyến. Tiền hoàn vào ví."); st.rerun()

        if st.session_state.sos_triggered:
            st.markdown("""
            <div style="background:rgba(255,0,0,0.15);border:2px solid #ff4444;border-radius:12px;padding:16px;margin-top:12px;text-align:center;">
                <div style="font-size:24px;font-weight:800;color:#ff4444;">🆘 TÍN HIỆU SOS ĐÃ GỬI!</div>
                <div style="color:#ff9999;margin-top:8px;">Gọi ngay: <b style="color:#ff4444;">1800 1234</b></div>
            </div>""", unsafe_allow_html=True)

        if not st.session_state.current_ride and st.session_state.history and not st.session_state.rating_submitted:
            last = st.session_state.history[0]
            st.markdown("---")
            st.markdown(f"### ⭐ Đánh giá chuyến với **{last['driver']}**")
            r_cols = st.columns(5)
            for i, col in enumerate(r_cols):
                if col.button("⭐"*(i+1),key=f"rate_{i+1}",use_container_width=True):
                    st.session_state.last_rating = i+1
                    st.session_state.rating_submitted = True
                    st.session_state.loyalty_points += (i+1)*10
                    if i+1==5: st.session_state.challenges_done.add("c3")
                    st.success(f"Cảm ơn! {i+1}⭐ cho {last['driver']}. +{(i+1)*10} điểm!")

# ──────────────────────────────────────────────────
# TAB 3 — LỊCH SỬ
# ──────────────────────────────────────────────────
with tab3:
    st.subheader("📜 Lịch sử chuyến đi")
    if not st.session_state.history:
        st.markdown("""<div style="text-align:center;padding:60px 20px;">
            <div style="font-size:48px;margin-bottom:16px;">🛵</div>
            <div style="color:#6a8f6a;font-size:16px;">Chưa có lịch sử chuyến đi nào</div></div>""", unsafe_allow_html=True)
    else:
        total_spent = sum(r["fare"] for r in st.session_state.history)
        total_km    = sum(r["dist"] for r in st.session_state.history)
        total_co2   = sum(r.get("co2_saved",0) for r in st.session_state.history)
        s1,s2,s3,s4 = st.columns(4)
        s1.metric("💰 Tổng chi",f"{total_spent:,}₫")
        s2.metric("📏 Tổng km", f"{total_km:.1f} km")
        s3.metric("🛵 Số chuyến",len(st.session_state.history))
        s4.metric("🌱 CO₂",     f"{total_co2:.2f}kg")
        st.markdown("---")
        for r in st.session_state.history:
            pet_tag = " 🐾" if r.get("pet_mode") else ""
            with st.expander(f"{r['vehicle']}{pet_tag}  ·  {r['pickup']} → {r['dropoff']}  ·  {r['fare']:,}₫  ·  {r.get('date','—')}"):
                c1,c2,c3,c4 = st.columns(4)
                c1.metric("Tài xế",r["driver"]); c2.metric("Km",f"{r['dist']} km")
                c3.metric("Phút",f"{r['mins']} phút"); c4.metric("Giá",f"{r['fare']:,}₫")
                st.caption(f"Mã: {r['id']} · Biển số: {r['plate']}")
                if r.get("co2_saved",0)>0: st.success(f"🌱 Tiết kiệm {r['co2_saved']}kg CO₂")
                if st.button("🔄 Đặt lại",key=f"rebook_{r['id']}"):
                    st.session_state["dropoff_prefill"]=r["dropoff"]
                    st.info("✅ Đã điền sẵn điểm đến!")

# ──────────────────────────────────────────────────
# TAB 4 — HỖ TRỢ AI
# ──────────────────────────────────────────────────
with tab4:
    st.markdown("""
    <div style="background:linear-gradient(135deg,#0d2818,#1a3a1a);border:1px solid rgba(0,177,79,0.3);border-radius:14px;padding:16px 20px;margin-bottom:16px;display:flex;gap:14px;align-items:center;">
        <div style="width:48px;height:48px;background:#00b14f;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:22px;box-shadow:0 0 16px rgba(0,177,79,0.5);">🤖</div>
        <div>
            <div style="font-weight:800;font-size:16px;color:#ffffff;">GrabBot AI</div>
            <div style="color:#81c784;font-size:13px;">Hỗ trợ 24/7 <span class="live-badge" style="margin-left:8px;">ONLINE</span></div>
        </div>
    </div>""", unsafe_allow_html=True)

    qr_buttons = ["💰 Giá cước?","🏷️ Mã giảm giá?","❌ Hủy chuyến?","💚 Nạp ví?","⚡ GrabElectric?","🐾 GrabPet?"]
    qr_cols = st.columns(3)
    for i,qr in enumerate(qr_buttons):
        if qr_cols[i%3].button(qr,use_container_width=True,key=f"qr_{qr}"):
            st.session_state.messages.append({"role":"user","content":qr})

    st.markdown("---")
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]): st.write(msg["content"])

    if prompt := st.chat_input("Hỏi bất cứ điều gì về Grab..."):
        st.session_state.messages.append({"role":"user","content":prompt})
        with st.chat_message("user"): st.write(prompt)

        ride = st.session_state.current_ride
        ride_ctx = (f"Đang có chuyến: {ride['driver']}, {ride['fare']:,}₫, {ride['pickup']}→{ride['dropoff']}." if ride else "Không có chuyến.")
        t_surge,a_surges = compute_total_surge(st.session_state.surge_rain,st.session_state.surge_traffic,st.session_state.surge_peak)
        pet_ctx = f"GrabPet Mode đang bật, loài: {st.session_state.pet_type}." if st.session_state.pet_mode else "GrabPet tắt."
        system = f"""Bạn là GrabBot — trợ lý AI của Grab VN. Khách: LeAnh. Số dư: {st.session_state.balance:,}₫. Chuyến: {st.session_state.trip_count}. Hạng: Gold.
{ride_ctx} Surge ×{t_surge}. {pet_ctx} Spin còn: {st.session_state.spin_available} lượt.
Trả lời ngắn gọn tiếng Việt, tối đa 3 câu, dùng emoji. Mã: GRAB20,NEWUSER,HCMC10,ECO15,FLASH30,PET10."""
        with st.chat_message("assistant"):
            with st.spinner("GrabBot đang trả lời..."):
                try:
                    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
                    resp = client.messages.create(model="claude-sonnet-4-20250514",max_tokens=300,system=system,
                        messages=[{"role":m["role"],"content":m["content"]} for m in st.session_state.messages[-8:]])
                    reply = resp.content[0].text
                except:
                    t = prompt.lower()
                    if "pet" in t or "thú cưng" in t: reply = "🐾 GrabPet Mode cho phép mang thú cưng! Bật ở sidebar, chọn GrabCar/Electric. Dùng mã PET10 giảm 10%!"
                    elif "vòng quay" in t or "spin" in t: reply = f"🎰 Bạn còn {st.session_state.spin_available} lượt quay! Vào tab Vòng Quay để nhận thưởng ngay nhé!"
                    elif "thử thách" in t: reply = "🎯 Hoàn thành Daily Challenge mỗi ngày để nhận điểm thưởng! Xem ở tab Thử Thách."
                    elif "đi chung" in t: reply = "👫 Grab Together cho phép bạn tạo phòng đi chung với bạn bè, chia tiền tự động! Vào tab Đi Chung."
                    elif "hủy" in t: reply = "Nhấn 'Hủy chuyến' trong tab Theo Dõi. Phí hủy có thể áp dụng nếu tài xế đã đến."
                    elif "giá" in t: reply = f"Giá = (Phí cơ bản + km×đơn giá + phút×đơn giá) × surge ×{t_surge}."
                    else: reply = "Cảm ơn đã liên hệ GrabBot! 🤖 Cho mình biết thêm để hỗ trợ bạn tốt hơn nhé."
                st.write(reply)
        st.session_state.messages.append({"role":"assistant","content":reply})

# ──────────────────────────────────────────────────
# TAB 5 — ƯU ĐÃI
# ──────────────────────────────────────────────────
with tab5:
    st.subheader("🎁 Ưu đãi & Flash Deals")
    st.markdown("""
    <div style="background:linear-gradient(135deg,#1a0a00,#2a1500);border:2px solid rgba(255,107,53,0.6);
         border-radius:16px;padding:20px 24px;margin-bottom:20px;text-align:center;">
        <div style="font-size:28px;margin-bottom:8px;">⚡ FLASH SALE HÔM NAY</div>
        <div style="display:flex;gap:12px;justify-content:center;flex-wrap:wrap;">
            <div style="background:rgba(255,107,53,0.2);border-radius:10px;padding:10px 16px;">
                <div style="color:#ff6b35;font-size:24px;font-weight:800;">02</div><div style="color:#ffcba4;font-size:11px;">GIỜ</div>
            </div>
            <div style="color:#ff6b35;font-size:24px;font-weight:800;line-height:52px;">:</div>
            <div style="background:rgba(255,107,53,0.2);border-radius:10px;padding:10px 16px;">
                <div style="color:#ff6b35;font-size:24px;font-weight:800;">34</div><div style="color:#ffcba4;font-size:11px;">PHÚT</div>
            </div>
        </div>
    </div>""", unsafe_allow_html=True)

    for deal in FLASH_DEALS:
        fd1,fd2 = st.columns([3,1])
        with fd1:
            st.markdown(f"""
            <div class="feature-card">
                <div style="display:flex;gap:14px;align-items:center;">
                    <div style="font-size:36px;">{deal['icon']}</div>
                    <div style="flex:1;">
                        <div style="font-size:18px;font-weight:700;color:#ffffff;">{deal['title']}</div>
                        <div style="color:#81c784;font-size:13px;">Còn <b style="color:#ff9800;">{deal['left']} lượt</b> · Hết {deal['expires']}</div>
                        <code style="background:#0f1f0f;color:#00e676;padding:3px 10px;border-radius:6px;font-size:14px;font-weight:700;">{deal['code']}</code>
                    </div>
                    <div style="color:#ff6b35;font-size:28px;font-weight:800;">-{deal['discount']}</div>
                </div>
            </div>""", unsafe_allow_html=True)
        with fd2:
            if st.button("Dùng ngay",key=f"use_{deal['code']}",use_container_width=True):
                st.success(f"✅ Mã **{deal['code']}** đã copy!")

    st.markdown("---")
    st.markdown("### 🏷️ Tất cả mã")
    for code, pct in PROMOS.items():
        p1,p2,p3 = st.columns([2,1,1])
        with p1:
            desc = {"GRAB20":"Giảm 20% mọi xe","NEWUSER":"Người dùng mới","HCMC10":"Khu vực HCMC",
                    "ECO15":"Xe điện & eco","FLASH30":"Flash sale cuối tuần","PET10":"GrabPet Mode 🐾"}.get(code,"")
            st.markdown(f"""<div style="background:#1a2a1a;border-radius:10px;padding:12px 16px;">
                <code style="font-size:16px;color:#00e676;font-weight:700;">{code}</code>
                <span class="promo-badge" style="margin-left:8px;">-{int(pct*100)}%</span>
                <div style="color:#81c784;font-size:12px;margin-top:4px;">{desc}</div>
            </div>""", unsafe_allow_html=True)
        with p2: st.metric("Giảm",f"{int(pct*100)}%")
        with p3: st.button("📋 Copy",key=f"copy_{code}",use_container_width=True)

# ──────────────────────────────────────────────────
# TAB 6 — 🎰 VÒNG QUAY MAY MẮN (NEW)
# ──────────────────────────────────────────────────
with tab6:
    st.markdown("""
    <div style="background:linear-gradient(135deg,#1a0a2e,#2e0a4a);border:2px solid rgba(180,100,255,0.5);
         border-radius:18px;padding:24px;margin-bottom:20px;text-align:center;">
        <div style="font-size:36px;margin-bottom:8px;">🎰</div>
        <div style="font-size:22px;font-weight:800;color:#ffffff;">Vòng Quay May Mắn</div>
        <div style="color:#ce93d8;font-size:14px;margin-top:6px;">
            Đặt xe → Nhận lượt quay → Trúng thưởng mỗi ngày!
        </div>
    </div>""", unsafe_allow_html=True)

    spin_col, prize_col = st.columns([1, 1])

    with spin_col:
        # Draw wheel as HTML canvas
        wheel_html = """
<canvas id="wheel" width="300" height="300" style="display:block;margin:0 auto;border-radius:50%;"></canvas>
<div style="text-align:center;margin-top:12px;">
    <div id="pointer" style="font-size:32px;margin-bottom:8px;">⬆️</div>
    <div id="result-box" style="background:rgba(180,100,255,0.15);border:1px solid rgba(180,100,255,0.4);
         border-radius:12px;padding:12px 20px;min-height:52px;display:flex;align-items:center;justify-content:center;">
        <span id="result-text" style="color:#ce93d8;font-size:14px;">Bấm quay để nhận thưởng!</span>
    </div>
</div>
<script>
const prizes=[
    {label:"GRAB20",desc:"Giảm 20%",   color:"#00b14f"},
    {label:"5K",   desc:"Cashback 5K", color:"#ff9800"},
    {label:"HCMC10",desc:"Giảm 10%",  color:"#2196f3"},
    {label:"2X PTS",desc:"Nhân đôi điểm",color:"#9c27b0"},
    {label:"FLASH30",desc:"Giảm 30%", color:"#f44336"},
    {label:"ECO15",desc:"Eco -15%",   color:"#4caf50"},
    {label:"10K",  desc:"Cashback 10K",color:"#ff5722"},
    {label:"JACKPOT",desc:"Giảm 50%!",color:"#ffd700"},
];
const canvas=document.getElementById("wheel");
const ctx=canvas.getContext("2d");
const N=prizes.length;
const arc=2*Math.PI/N;
let currentAngle=0;
let spinning=false;

function drawWheel(angle){
    ctx.clearRect(0,0,300,300);
    for(let i=0;i<N;i++){
        const start=angle+i*arc;
        ctx.beginPath();
        ctx.moveTo(150,150);
        ctx.arc(150,150,140,start,start+arc);
        ctx.closePath();
        ctx.fillStyle=prizes[i].color;
        ctx.fill();
        ctx.strokeStyle="#0a0e0b";
        ctx.lineWidth=2;
        ctx.stroke();
        ctx.save();
        ctx.translate(150,150);
        ctx.rotate(start+arc/2);
        ctx.textAlign="right";
        ctx.fillStyle="#ffffff";
        ctx.font="bold 13px 'Be Vietnam Pro',sans-serif";
        ctx.fillText(prizes[i].label,128,5);
        ctx.restore();
    }
    ctx.beginPath();
    ctx.arc(150,150,22,0,2*Math.PI);
    ctx.fillStyle="#0a0e0b";
    ctx.fill();
    ctx.strokeStyle="#00b14f";
    ctx.lineWidth=3;
    ctx.stroke();
}

drawWheel(currentAngle);

function spin(winIdx){
    if(spinning)return;
    spinning=true;
    document.getElementById("result-text").innerText="Đang quay...";
    const totalRotation=5*2*Math.PI + (2*Math.PI - winIdx*arc - arc/2 - currentAngle%(2*Math.PI));
    const duration=3000;
    const start=performance.now();
    const startAngle=currentAngle;
    function animate(now){
        const elapsed=Math.min((now-start)/duration,1);
        const ease=1-Math.pow(1-elapsed,4);
        currentAngle=startAngle+totalRotation*ease;
        drawWheel(currentAngle);
        if(elapsed<1){requestAnimationFrame(animate);}
        else{
            spinning=false;
            const prize=prizes[winIdx];
            document.getElementById("result-text").innerHTML=
                '<span style="color:#ffd700;font-size:18px;font-weight:800;">🎉 '+prize.label+'</span><br>'+
                '<span style="color:#ce93d8;font-size:13px;">'+prize.desc+'</span>';
            window.spinResult=prize.label;
        }
    }
    requestAnimationFrame(animate);
}

window.doSpin=spin;
drawWheel(0);
</script>"""
        st.components.v1.html(wheel_html, height=420)

    with prize_col:
        avail = st.session_state.spin_available
        st.markdown(f"""
        <div style="background:rgba(180,100,255,0.12);border:1px solid rgba(180,100,255,0.4);
             border-radius:14px;padding:20px;text-align:center;margin-bottom:16px;">
            <div style="color:#ce93d8;font-size:13px;text-transform:uppercase;letter-spacing:1px;">Lượt quay của bạn</div>
            <div style="color:#ffffff;font-size:48px;font-weight:800;">{avail}</div>
            <div style="color:#ce93d8;font-size:12px;margin-top:4px;">Đặt xe để nhận thêm lượt quay</div>
        </div>""", unsafe_allow_html=True)

        st.markdown("**🎁 Bảng giải thưởng:**")
        for p in SPIN_PRIZES:
            pct = int(p["prob"]*100)
            st.markdown(f"""
            <div style="display:flex;align-items:center;gap:10px;padding:6px 0;border-bottom:1px solid rgba(255,255,255,0.05);">
                <div style="width:12px;height:12px;border-radius:50%;background:{p['color']};flex-shrink:0;"></div>
                <span style="color:#ffffff;font-weight:700;font-size:13px;flex:1;">{p['label']}</span>
                <span style="color:#ce93d8;font-size:12px;">{p['desc']}</span>
                <span style="color:#888;font-size:11px;min-width:30px;text-align:right;">{pct}%</span>
            </div>""", unsafe_allow_html=True)

        st.markdown("---")
        if avail > 0:
            if st.button("🎰 QUAY NGAY!", use_container_width=True, key="spin_btn"):
                # Pick winner by probability
                rand = random.random(); cumul = 0; win_idx = 0
                for i, p in enumerate(SPIN_PRIZES):
                    cumul += p["prob"]
                    if rand <= cumul: win_idx = i; break
                prize = SPIN_PRIZES[win_idx]
                st.session_state.spin_available -= 1
                st.session_state.spin_history.insert(0,{
                    "prize": prize["label"], "desc": prize["desc"],
                    "date": datetime.now().strftime("%d/%m %H:%M")
                })
                if prize["label"] == "2X PTS":
                    st.session_state.loyalty_points = int(st.session_state.loyalty_points * 1.5)
                elif prize["label"] in ("5K","10K"):
                    amt = 5000 if prize["label"]=="5K" else 10000
                    st.session_state.balance += amt
                st.success(f"🎉 Bạn trúng: **{prize['label']}** — {prize['desc']}!")

                # Show JS spin animation trigger
                spin_trigger = f"""<script>
                setTimeout(function(){{if(typeof window.doSpin==='function'){{window.doSpin({win_idx});}}else{{console.log('spin fn not ready');}}}},200);
                </script>"""
                st.components.v1.html(spin_trigger, height=0)
        else:
            st.markdown("""
            <div style="background:rgba(255,255,255,0.05);border-radius:12px;padding:16px;text-align:center;">
                <div style="color:#888;font-size:14px;">Hết lượt quay hôm nay 😢</div>
                <div style="color:#81c784;font-size:12px;margin-top:6px;">Đặt thêm chuyến để nhận lượt mới!</div>
            </div>""", unsafe_allow_html=True)

    if st.session_state.spin_history:
        st.markdown("---")
        st.markdown("### 📋 Lịch sử giải thưởng")
        for h in st.session_state.spin_history[:6]:
            st.markdown(f"""
            <div style="display:flex;align-items:center;justify-content:space-between;padding:8px 14px;
                 background:#1a1a2e;border-radius:8px;margin:4px 0;border:1px solid rgba(180,100,255,0.2);">
                <span style="color:#ffd700;font-weight:700;">🎰 {h['prize']}</span>
                <span style="color:#ce93d8;font-size:13px;">{h['desc']}</span>
                <span style="color:#666;font-size:12px;">{h['date']}</span>
            </div>""", unsafe_allow_html=True)

# ──────────────────────────────────────────────────
# TAB 7 — 🎯 THỬ THÁCH HẰNG NGÀY (NEW)
# ──────────────────────────────────────────────────
with tab7:
    now_d = datetime.now()
    st.markdown(f"""
    <div style="background:linear-gradient(135deg,#1a1a0a,#2a2a10);border:2px solid rgba(255,215,0,0.4);
         border-radius:18px;padding:20px 24px;margin-bottom:20px;display:flex;align-items:center;gap:16px;">
        <span style="font-size:40px;">🎯</span>
        <div>
            <div style="font-size:20px;font-weight:800;color:#ffffff;">Daily Challenge</div>
            <div style="color:#ffd54f;font-size:14px;">Ngày {now_d.strftime('%d/%m/%Y')} — Reset lúc 00:00</div>
            <div style="color:#ffb74d;font-size:13px;margin-top:3px;">
                Hoàn thành nhiệm vụ để nhận điểm & phần thưởng đặc biệt!
            </div>
        </div>
    </div>""", unsafe_allow_html=True)

    done_count = len(st.session_state.challenges_done)
    total_challs = len(DAILY_CHALLENGES)
    prog_pct = done_count / total_challs
    total_pts_earned = sum(c["pts"] for c in DAILY_CHALLENGES if c["id"] in st.session_state.challenges_done)

    cp1, cp2, cp3 = st.columns(3)
    cp1.metric("✅ Hoàn thành", f"{done_count}/{total_challs}")
    cp2.metric("🎯 Điểm hôm nay", f"+{total_pts_earned}")
    cp3.metric("🔥 Streak",       f"{st.session_state.streak_days} ngày")

    st.markdown("**Tiến độ hôm nay:**")
    st.progress(prog_pct)
    st.caption(f"{done_count}/{total_challs} nhiệm vụ · {int(prog_pct*100)}% hoàn thành")

    if done_count == total_challs:
        st.markdown("""
        <div style="background:linear-gradient(135deg,#0d2818,#1a4a1a);border:2px solid #00e676;
             border-radius:14px;padding:16px;margin:12px 0;text-align:center;">
            <div style="font-size:28px;">🏆</div>
            <div style="color:#00e676;font-weight:800;font-size:16px;margin-top:6px;">PERFECT DAY! Hoàn thành tất cả!</div>
            <div style="color:#81c784;font-size:13px;margin-top:4px;">+200 điểm bonus + 🎰 2 lượt quay thêm!</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 📋 Nhiệm vụ hôm nay")

    for chall in DAILY_CHALLENGES:
        done = chall["id"] in st.session_state.challenges_done
        card_class = "challenge-card challenge-done" if done else "challenge-card"
        border_col = "rgba(0,177,79,0.5)" if done else "rgba(100,100,255,0.3)"
        bg_col     = "linear-gradient(135deg,#0d2818,#1a3a1a)" if done else "linear-gradient(135deg,#1a1a2e,#16213e)"
        check_icon = "✅" if done else "⬜"
        pts_col    = "#00e676" if done else "#7986cb"

        cc1, cc2 = st.columns([4, 1])
        with cc1:
            st.markdown(f"""
            <div style="background:{bg_col};border:1px solid {border_col};border-radius:14px;padding:16px 18px;margin:6px 0;">
                <div style="display:flex;align-items:center;gap:14px;">
                    <span style="font-size:28px;">{chall['icon']}</span>
                    <div style="flex:1;">
                        <div style="display:flex;align-items:center;gap:8px;">
                            <span style="color:#ffffff;font-weight:700;font-size:15px;">{chall['title']}</span>
                            <span style="font-size:16px;">{check_icon}</span>
                        </div>
                        <div style="color:#a5d6a7;font-size:13px;margin-top:3px;">{chall['desc']}</div>
                    </div>
                    <div style="text-align:center;">
                        <div style="color:{pts_col};font-weight:800;font-size:18px;">+{chall['pts']}</div>
                        <div style="color:#888;font-size:11px;">điểm</div>
                        <div style="color:#7986cb;font-size:11px;margin-top:2px;">+{chall['xp']} XP</div>
                    </div>
                </div>
            </div>""", unsafe_allow_html=True)
        with cc2:
            if not done:
                if st.button("Nhận",key=f"chall_claim_{chall['id']}",use_container_width=True):
                    st.warning("Hoàn thành nhiệm vụ trước để nhận điểm!")
            else:
                st.markdown('<div style="text-align:center;padding:18px 0;color:#00e676;font-size:20px;">✓</div>', unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 🏅 Thành tích streak")
    streak_milestones = [
        {"days":3,"reward":"🎁 Mã GRAB20","unlocked":st.session_state.streak_days>=3},
        {"days":7,"reward":"🎰 3 lượt quay","unlocked":st.session_state.streak_days>=7},
        {"days":14,"reward":"⭐ Lên hạng Platinum","unlocked":st.session_state.streak_days>=14},
        {"days":30,"reward":"💎 Cashback 50K","unlocked":st.session_state.streak_days>=30},
    ]
    ms_cols = st.columns(len(streak_milestones))
    for col, ms in zip(ms_cols, streak_milestones):
        col.markdown(f"""
        <div style="background:{'rgba(0,177,79,0.15)' if ms['unlocked'] else '#141f14'};
             border:1px solid {'rgba(0,177,79,0.5)' if ms['unlocked'] else 'rgba(255,255,255,0.08)'};
             border-radius:12px;padding:14px 10px;text-align:center;opacity:{'1' if ms['unlocked'] else '0.6'};">
            <div style="font-size:22px;">{'🔓' if ms['unlocked'] else '🔒'}</div>
            <div style="color:{'#00e676' if ms['unlocked'] else '#888'};font-size:13px;font-weight:700;margin-top:6px;">{ms['days']} ngày</div>
            <div style="color:#a5d6a7;font-size:11px;margin-top:4px;">{ms['reward']}</div>
        </div>""", unsafe_allow_html=True)

# ──────────────────────────────────────────────────
# TAB 8 — 👫 GRAB TOGETHER (NEW)
# ──────────────────────────────────────────────────
with tab8:
    st.markdown("""
    <div style="background:linear-gradient(135deg,#0a1a2e,#1a2a4a);border:2px solid rgba(33,150,243,0.5);
         border-radius:18px;padding:24px;margin-bottom:20px;display:flex;align-items:center;gap:16px;">
        <span style="font-size:40px;">👫</span>
        <div>
            <div style="font-size:20px;font-weight:800;color:#ffffff;">Grab Together</div>
            <div style="color:#90caf9;font-size:14px;">Tạo phòng đi chung · Chia tiền tự động · Chat real-time</div>
        </div>
    </div>""", unsafe_allow_html=True)

    tog_col1, tog_col2 = st.columns([1,1])

    with tog_col1:
        st.markdown("### 🏠 Tạo phòng mới")
        room_name    = st.text_input("Tên phòng", placeholder="VD: Team đi ăn trưa", key="tog_room_name")
        room_dest    = st.text_input("Điểm đến chung", placeholder="VD: Landmark 81", key="tog_dest")
        room_vehicle = st.selectbox("Loại xe", [k for k,v in VEHICLES.items() if v["seats"]>=2], key="tog_vehicle")
        max_members  = st.slider("Số người tối đa", 2, VEHICLES[room_vehicle]["seats"], 2, key="tog_max")
        split_method = st.selectbox("Cách chia tiền", ["Chia đều","Người đặt trả trước","Mỗi người tự trả"], key="tog_split")

        if st.button("🚀 Tạo Phòng Grab Together", use_container_width=True, key="create_tog"):
            if not room_name or not room_dest:
                st.error("Vui lòng nhập tên phòng và điểm đến!")
            else:
                room_code = f"GT{random.randint(1000,9999)}"
                st.session_state.together_room = {
                    "code": room_code, "name": room_name, "dest": room_dest,
                    "vehicle": room_vehicle, "max": max_members,
                    "split": split_method, "members": ["LeAnh (bạn)"],
                    "created": datetime.now().strftime("%H:%M"),
                    "status": "Chờ thành viên",
                }
                st.session_state.challenges_done.add("c5")
                st.success(f"✅ Đã tạo phòng! Mã: **{room_code}**")
                st.rerun()

    with tog_col2:
        st.markdown("### 🔑 Tham gia phòng")
        join_code = st.text_input("Nhập mã phòng", placeholder="VD: GT1234", key="tog_join_code")
        your_name = st.text_input("Tên của bạn", placeholder="VD: An", key="tog_your_name")

        if st.button("🚪 Tham Gia", use_container_width=True, key="join_tog"):
            if st.session_state.together_room and join_code == st.session_state.together_room["code"]:
                room = st.session_state.together_room
                member_name = your_name or f"Thành viên {len(room['members'])+1}"
                if len(room["members"]) < room["max"]:
                    room["members"].append(member_name)
                    st.success(f"✅ {member_name} đã vào phòng **{room['name']}**!")
                    st.rerun()
                else:
                    st.error("Phòng đã đầy!")
            else:
                st.error("Mã phòng không đúng hoặc chưa có phòng nào được tạo.")

    if st.session_state.together_room:
        room = st.session_state.together_room
        st.markdown("---")
        st.markdown("### 🏠 Phòng hiện tại")

        member_count = len(room["members"])
        est_dist = random.uniform(4,12)
        est_fare = round((VEHICLES[room["vehicle"]]["base"] +
                         VEHICLES[room["vehicle"]]["per_km"]*est_dist +
                         VEHICLES[room["vehicle"]]["per_min"]*15) / 1000) * 1000
        per_person = round(est_fare / member_count / 1000) * 1000

        st.markdown(f"""
        <div style="background:linear-gradient(135deg,#0a1a2e,#1a2a4a);border:2px solid rgba(33,150,243,0.4);
             border-radius:16px;padding:20px 24px;">
            <div style="display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:12px;">
                <div>
                    <div style="font-size:20px;font-weight:800;color:#ffffff;">{room['name']}</div>
                    <div style="color:#90caf9;font-size:14px;margin-top:4px;">
                        🏁 {room['dest']} · {room['vehicle']}
                    </div>
                    <div style="color:#64b5f6;font-size:13px;margin-top:3px;">
                        Chia tiền: {room['split']} · Tạo lúc {room['created']}
                    </div>
                </div>
                <div style="background:rgba(33,150,243,0.2);border:1px solid rgba(33,150,243,0.5);
                     border-radius:10px;padding:8px 16px;text-align:center;">
                    <div style="color:#64b5f6;font-size:11px;">Mã phòng</div>
                    <div style="color:#ffffff;font-size:22px;font-weight:800;font-family:monospace;">{room['code']}</div>
                </div>
            </div>
            <div style="margin-top:16px;padding-top:16px;border-top:1px solid rgba(33,150,243,0.2);">
                <div style="color:#90caf9;font-size:13px;margin-bottom:10px;">
                    👥 Thành viên ({member_count}/{room['max']}):
                </div>
                <div style="display:flex;gap:10px;flex-wrap:wrap;">""", unsafe_allow_html=True)

        member_badges = "".join([f"""
                    <div style="background:rgba(33,150,243,0.15);border:1px solid rgba(33,150,243,0.4);
                         border-radius:20px;padding:6px 14px;display:flex;align-items:center;gap:8px;">
                        <div style="width:28px;height:28px;background:linear-gradient(135deg,#1565c0,#0d47a1);
                             border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:13px;font-weight:700;color:#fff;">
                            {m[0].upper()}</div>
                        <span style="color:#e3f2fd;font-size:13px;">{m}</span>
                    </div>""" for m in room["members"]])

        st.markdown(member_badges + "</div></div></div>", unsafe_allow_html=True)

        # Slots remaining
        remaining = room["max"] - member_count
        if remaining > 0:
            st.markdown(f"""
            <div style="background:rgba(33,150,243,0.08);border:1px solid rgba(33,150,243,0.2);
                 border-radius:10px;padding:10px 16px;margin-top:10px;text-align:center;">
                <span style="color:#90caf9;font-size:13px;">Còn {remaining} chỗ trống · Chia sẻ mã <b>{room['code']}</b> cho bạn bè</span>
            </div>""", unsafe_allow_html=True)

        st.markdown("---")
        f1,f2,f3 = st.columns(3)
        f1.metric("💰 Ước tính tổng", f"{est_fare:,}₫")
        f2.metric("👤 Mỗi người",     f"{per_person:,}₫")
        f3.metric("👥 Thành viên",    f"{member_count}/{room['max']}")

        st.markdown("---")
        btn1,btn2,btn3 = st.columns(3)
        with btn1:
            if st.button("🚕 Đặt xe cho nhóm",use_container_width=True,key="tog_book"):
                st.success("✅ Đang tìm xe cho cả nhóm! Chuyển sang tab Theo Dõi.")
        with btn2:
            if st.button("💬 Chat nhóm",use_container_width=True,key="tog_chat"):
                st.info("📱 Tính năng chat nhóm đang phát triển!")
        with btn3:
            if st.button("🗑️ Đóng phòng",use_container_width=True,key="tog_close"):
                st.session_state.together_room = None; st.rerun()

        st.markdown("### 💬 Nhắn tin nhanh")
        quick_msgs = ["Tôi đang trên đường 🏃","5 phút nữa tới 👋","Chờ mình ở cổng 🚪","OK tôi sẵn sàng ✅"]
        qm_cols = st.columns(2)
        for i,msg in enumerate(quick_msgs):
            if qm_cols[i%2].button(msg,key=f"qm_{i}",use_container_width=True):
                st.success(f'📢 Đã gửi: "{msg}"')

# ──────────────────────────────────────────────────
# TAB 9 — 💳 VÍ & CHI TIÊU ANALYTICS (NEW)
# ──────────────────────────────────────────────────
with tab9:
    st.markdown("""
    <div style="background:linear-gradient(135deg,#0a1a0a,#1a2e1a);border:2px solid rgba(0,177,79,0.4);
         border-radius:18px;padding:24px;margin-bottom:20px;display:flex;align-items:center;gap:16px;">
        <span style="font-size:40px;">💳</span>
        <div>
            <div style="font-size:20px;font-weight:800;color:#ffffff;">Grab Wallet Analytics</div>
            <div style="color:#81c784;font-size:14px;">Theo dõi chi tiêu thông minh · Biểu đồ · Báo cáo</div>
        </div>
    </div>""", unsafe_allow_html=True)

    # Balance card
    st.markdown(f"""
    <div style="background:linear-gradient(135deg,#0d2818,#1a3a1a);border:1px solid rgba(0,177,79,0.4);
         border-radius:16px;padding:20px 24px;margin-bottom:16px;">
        <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:12px;">
            <div>
                <div style="color:#a5d6a7;font-size:13px;text-transform:uppercase;letter-spacing:1px;">Số dư GrabPay</div>
                <div style="font-size:36px;font-weight:800;color:#00e676;margin-top:4px;">{st.session_state.balance:,} ₫</div>
            </div>
            <div style="text-align:right;">
                <div style="color:#a5d6a7;font-size:12px;">Điểm thưởng</div>
                <div style="color:#ffd54f;font-size:22px;font-weight:800;">{st.session_state.loyalty_points:,} pts</div>
            </div>
        </div>
    </div>""", unsafe_allow_html=True)

    # ── NEW: Price forecast chart ──
    st.markdown("### 📈 Dự báo giá surge 24 giờ")
    st.markdown("""<div style="color:#81c784;font-size:13px;margin-bottom:10px;">
        Biểu đồ dự báo hệ số surge theo giờ trong ngày — chọn giờ thấp để tiết kiệm nhất!</div>""",
        unsafe_allow_html=True)

    forecast = st.session_state.surge_forecast
    now_h = datetime.now().hour
    bars = ""
    for i in range(24):
        val = forecast[i]
        if val >= 1.4:
            bar_color = "#ff5555"
            txt_color = "#ff9a6c"
        elif val >= 1.2:
            bar_color = "#ff9800"
            txt_color = "#ffd54f"
        else:
            bar_color = "#00b14f"
            txt_color = "#81c784"
        highlight = "box-shadow:0 0 8px rgba(0,177,79,0.6);border:2px solid #00e676;" if i == now_h else ""
        opacity = "1" if i == now_h else "0.7"
        hour_color = "#ffffff" if i == now_h else "#666"
        hour_weight = "700" if i == now_h else "400"
        bar_height = int((val - 0.7) * 120)
        bars += f"""
        <div style="flex:1;display:flex;flex-direction:column;align-items:center;gap:4px;">
            <div style="font-size:8px;color:{txt_color};font-weight:700;">{val:.1f}×</div>
            <div style="width:100%;border-radius:4px 4px 0 0;background:{bar_color};height:{bar_height}px;{highlight}opacity:{opacity};"></div>
            <div style="font-size:8px;color:{hour_color};font-weight:{hour_weight};">{i:02d}h</div>
        </div>"""

    chart_html = f"""
    <div style="position:relative;height:220px;display:flex;align-items:flex-end;gap:3px;padding:0 4px;margin-bottom:8px;">
        {bars}
    </div>
    <div style="display:flex;gap:16px;font-size:12px;color:#a5d6a7;justify-content:center;margin-top:8px;">
        <span>🟢 Giá thường (&lt;1.2×)</span>
        <span>🟡 Surge nhẹ (1.2–1.4×)</span>
        <span>🔴 Surge cao (≥1.4×)</span>
        <span style="color:#00e676;">▌ Hiện tại ({now_h:02d}h)</span>
    </div>"""

    st.components.v1.html(
        f'<div style="background:transparent;font-family:sans-serif;">{chart_html}</div>',
        height=260
    )

    # Best hours to book
    sorted_hours = sorted(range(24), key=lambda h: forecast[h])
    best3 = sorted_hours[:3]
    worst3 = sorted_hours[-3:]
    bh1,bh2 = st.columns(2)
    with bh1:
        st.markdown(f"""
        <div style="background:rgba(0,177,79,0.1);border:1px solid rgba(0,177,79,0.3);border-radius:12px;padding:14px 16px;">
            <div style="color:#00e676;font-weight:700;font-size:14px;margin-bottom:8px;">✅ Giờ tốt nhất để đặt</div>
            {''.join([f'<div style="color:#a5d6a7;font-size:13px;margin:4px 0;">🕐 {h:02d}:00 — ×{forecast[h]:.2f}</div>' for h in best3])}
        </div>""", unsafe_allow_html=True)
    with bh2:
        st.markdown(f"""
        <div style="background:rgba(255,80,50,0.08);border:1px solid rgba(255,80,50,0.3);border-radius:12px;padding:14px 16px;">
            <div style="color:#ff9a6c;font-weight:700;font-size:14px;margin-bottom:8px;">⚠️ Giờ surge cao nhất</div>
            {''.join([f'<div style="color:#ffcba4;font-size:13px;margin:4px 0;">🕐 {h:02d}:00 — ×{forecast[h]:.2f}</div>' for h in worst3])}
        </div>""", unsafe_allow_html=True)

    st.markdown("---")

    # ── Spending breakdown ──
    st.markdown("### 💸 Phân tích chi tiêu")

    period = st.selectbox("Xem theo", ["7 ngày gần nhất","Tháng này","Tháng trước"], key="wallet_period")
    txns = st.session_state.wallet_txns
    expenses = [t for t in txns if t["amt"] < 0]
    incomes  = [t for t in txns if t["amt"] > 0]
    total_out = abs(sum(t["amt"] for t in expenses))
    total_in  = sum(t["amt"] for t in incomes)

    wa1,wa2,wa3 = st.columns(3)
    wa1.metric("💸 Tổng chi",  f"{total_out:,}₫", f"-{total_out//len(expenses):,}₫ bình quân")
    wa2.metric("💚 Tổng nạp",  f"{total_in:,}₫")
    wa3.metric("📊 Số GD",     len(txns))

    # Category breakdown
    cat_totals = {}
    for t in expenses:
        cat_totals[t["cat"]] = cat_totals.get(t["cat"], 0) + abs(t["amt"])
    cat_names = {"🚲":"GrabBike","🚗":"GrabCar","🚙":"GrabCar 7c","⚡":"GrabElectric","🐾":"GrabPet"}
    st.markdown("**Chi tiêu theo loại xe:**")
    for cat, amt in sorted(cat_totals.items(), key=lambda x: -x[1]):
        pct = int(amt / total_out * 100) if total_out else 0
        name = cat_names.get(cat, cat)
        st.markdown(f"""
        <div style="display:flex;align-items:center;gap:12px;margin:8px 0;">
            <span style="font-size:18px;min-width:24px;">{cat}</span>
            <span style="color:#c8e6c9;font-size:13px;min-width:100px;">{name}</span>
            <div style="flex:1;background:#1a2a1a;border-radius:4px;height:12px;overflow:hidden;">
                <div class="wallet-bar" style="width:{pct}%;height:100%;"></div>
            </div>
            <span style="color:#00e676;font-size:13px;min-width:80px;text-align:right;">{amt:,}₫</span>
            <span style="color:#666;font-size:12px;min-width:30px;">{pct}%</span>
        </div>""", unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### 📋 Lịch sử giao dịch")

    for t in txns:
        amt_color = "#00e676" if t["amt"] > 0 else "#ff6b6b"
        amt_sign  = "+" if t["amt"] > 0 else ""
        st.markdown(f"""
        <div style="display:flex;align-items:center;justify-content:space-between;padding:10px 14px;
             background:#141f14;border-radius:10px;margin:4px 0;border:1px solid rgba(0,177,79,0.15);">
            <div style="display:flex;align-items:center;gap:12px;">
                <span style="font-size:20px;">{t['cat']}</span>
                <div>
                    <div style="color:#ffffff;font-size:14px;font-weight:500;">{t['desc']}</div>
                    <div style="color:#81c784;font-size:12px;">{t['date']}</div>
                </div>
            </div>
            <div style="color:{amt_color};font-weight:700;font-size:15px;">{amt_sign}{t['amt']:,}₫</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("---")
    nap1,nap2 = st.columns(2)
    with nap1:
        nap_amt = st.selectbox("💚 Nạp ví GrabPay", ["50,000₫","100,000₫","200,000₫","500,000₫","1,000,000₫"], key="topup_amt")
    with nap2:
        st.markdown("<div style='height:28px'></div>",unsafe_allow_html=True)
        if st.button("⚡ Nạp ngay",use_container_width=True,key="topup_btn"):
            amt = int(nap_amt.replace(",","").replace("₫",""))
            st.session_state.balance += amt
            st.session_state.wallet_txns.insert(0,{"date":datetime.now().strftime("%d/%m"),"desc":"GrabPay top-up","amt":amt,"cat":"💚"})
            st.success(f"✅ Đã nạp **{amt:,}₫** vào ví GrabPay!")
            st.rerun()

Overwriting app.py


In [11]:
# ══════════════════════════════════════════
# CELL 4 — Chạy app + tạo link public
# ══════════════════════════════════════════
NGROK_TOKEN = "3DFvCJVYwIGdAyqwiLEDuaBONez_29crAqnArhG8sbtsvXpC3"   # ← THAY VÀO ĐÂY

from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = NGROK_TOKEN

# Kill process cũ
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)
ngrok.kill()

# Khởi động Streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)

# Tạo link public
tunnel = ngrok.connect(8501)
print("\n" + "="*50)
print("🚀 APP ĐANG CHẠY!")
print(f"🌐 Link: {tunnel.public_url}")
print("="*50)
print("\n⚠️  Giữ tab Colab mở để app tiếp tục chạy")


🚀 APP ĐANG CHẠY!
🌐 Link: https://bruising-slideshow-chapped.ngrok-free.dev

⚠️  Giữ tab Colab mở để app tiếp tục chạy
